# DiT experiments — all five pruned variants

**Upload this notebook alone and run top to bottom.** This teammate runs **five pruned variants plus an original baseline**. The other notebook runs the five damped variants. Both include the feature-before/after trials and all three fixed position trials.

Sections 1–5 prepare the shared pipeline; section 6 defines metrics; sections 7A–7D generate only this notebook's assigned mode, display diagnostics, and compute LPIPS/FID. Section 8 exports results. Section 9 optionally compares the two teammates' completed run directories without new DiT inference.

**Shared selection:** each notebook can independently calibrate the same deterministic protocol. For guaranteed identical feature-block selection across different hardware, one teammate can share `RUN_ROOT/pairing_experiments/design.json`; the other sets `SHARED_PAIRING_DESIGN` in section 4 before first generation. This also skips duplicate calibration. Imported settings and code are validated. The comparison rejects mismatched selections.

Each notebook has a separate output prefix and its own baseline, so neither needs the other's generated images to run or evaluate. Keep protocol settings and metric package versions aligned. After both runs finish, copy/mount both output directories in one runtime for section 9. Comparison recomputes both LPIPS scores against the **same original baseline** to avoid cross-device baseline drift.

Validation defaults to 20 images per model. To run the frozen held-out split, set `PAIRING_SPLIT="test"` in 7B and rerun 7B–7D (5,000 images per model by default). Finish generation/diagnostics before metrics install optional packages. Resume using this notebook's own `RESUME_RUN`; runtime changes may require a fresh prepared run.


## 1. SETUP — workspace and dependencies

Run once per fresh runtime. Keep the runtime’s CUDA-compatible PyTorch/torchvision pair. If pip requests a runtime restart, restart and rerun from here. Drive mounting is optional; source stays in the working directory and outputs can be stored on Drive.

In [20]:
# SETUP — no experiment starts here.
from pathlib import Path
import os, sys, subprocess
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
WORKSPACE = Path.cwd()
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
OUTPUT_BASE = (Path("/content/drive/MyDrive/DiT_team_runs")
               if USE_GOOGLE_DRIVE else WORKSPACE / "team_runs")
subprocess.run([sys.executable, "-m", "pip", "install",
                "timm", "diffusers", "accelerate", "lpips", "numpy",
                "pandas", "matplotlib", "pillow", "pytest", "huggingface_hub"], check=True)
print("Source workspace:", WORKSPACE)
print("Output base:", OUTPUT_BASE)


Source workspace: /Users/matthewlee/Desktop/CORNELL/CS5787/cs5787-deep-learning
Output base: /Users/matthewlee/Desktop/CORNELL/CS5787/cs5787-deep-learning/team_runs



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


## 2. DEFINITIONS — execute the team code directly

Run every definition cell below in order. These are ordinary Python functions and classes in the notebook kernel: edit the code directly, then rerun the cell. No module extraction, package imports, or CLI subprocesses are used for experiments.

Definitions do not run inference. After a shared-code change, prepare a new run; source fingerprints detect executed changes. Unexecuted edits do not affect the kernel. After a runtime restart, rerun setup and definitions, then tests and configuration. Run state and protocol are saved as JSON artifacts.

### 2.1. DEFINE — protocol validation, fixed noise and image preprocessing

In [21]:
"""The team's input and image contract. Sampling uses label/seed pairs; real references are loaded only for evaluation."""

import hashlib
import json
from numbers import Integral
from pathlib import Path

import torch


def validate_pairs(pairs):
    """Return nonempty, unique ImageNet (label, seed) pairs without silent coercion."""
    result = []
    for pair in pairs:
        if len(pair) != 2:
            raise ValueError("Each sample must contain a label and seed")
        label, seed = pair
        if any(isinstance(v, bool) or not isinstance(v, Integral) for v in pair):
            raise ValueError("Labels and seeds must be integers")
        if not 0 <= label < 1000 or not 0 <= seed < 2**63 - 1_000_000:
            raise ValueError("Invalid ImageNet label or seed")
        result.append((int(label), int(seed)))
    if not result or len(set(result)) != len(result):
        raise ValueError("Sample pairs must be nonempty and unique")
    if len({s for _, s in result}) != len(result):
        raise ValueError("Use a different seed for every image")
    return result


def make_protocol(labels=None,
                  calibration_per_class=2, validation_per_class=2, test_per_class=8):
    """Balanced pilot with three disjoint seed sets; all members use this file."""
    labels = list(CLASS_LABELS if labels is None else labels)
    if not labels or len(set(labels)) != len(labels):
        raise ValueError("Use nonempty, unique class labels")
    counts = [calibration_per_class, validation_per_class, test_per_class]
    if any(isinstance(n, bool) or not isinstance(n, Integral) or n <= 0 for n in counts):
        raise ValueError("Samples per class must be positive integers")
    splits = {}
    next_seed = 10_000
    for name, count in zip(["calibration", "validation", "test"], counts):
        pairs = [(label, next_seed + i) for i, label in
                 enumerate(label for _ in range(count) for label in labels)]
        splits[name] = [list(pair) for pair in validate_pairs(pairs)]
        next_seed += len(pairs)
    return dict(version=1, settings=dict(image_size=256, architecture="DiT-XL/2",
                sampling_steps=50, cfg_scale=4.0, sampler="official ancestral p_sample_loop",
                dtype="float32", tf32=False, deterministic_algorithms=True,
                batch_size_images=1, initial_noise_device="cpu", sampler_seed_offset=1_000_000,
                vae_id="stabilityai/sd-vae-ft-mse",
                vae_revision="31f26fdeee1355a5c34592e401dd41e45d25a493",
                dit_commit="ed81ce2229091fd4ecc9a223645f95cf379d582b",
                checkpoint_sha256="9ec1876e4c03471bca126663a30e2d1b20610b6d2f87850a39a36f25cc685521"),
                splits=splits)


def load_protocol(path):
    protocol = (json.loads(json.dumps(path)) if isinstance(path, dict)
                else json.loads(Path(path).read_text()))
    if protocol.get("version") != 1 or set(protocol.get("splits", {})) != {"calibration", "validation", "test"}:
        raise ValueError("Expected protocol version 1 with calibration, validation and test splits")
    all_seeds = set()
    for name, pairs in protocol["splits"].items():
        pairs = validate_pairs(pairs)
        seeds = {seed for _, seed in pairs}
        if all_seeds & seeds:
            raise ValueError(f"Seed leakage into {name}")
        all_seeds |= seeds
    settings = protocol["settings"]
    if settings["image_size"] != 256 or settings["architecture"] != "DiT-XL/2":
        raise ValueError("This protocol supports DiT-XL/2 at 256 pixels")
    if settings.get("tf32") is not False or settings.get("deterministic_algorithms") is not True:
        raise ValueError("Use deterministic FP32 settings with TF32 disabled")
    if settings["dtype"] != "float32" or settings["sampler"] != "official ancestral p_sample_loop":
        raise ValueError("Unsupported dtype or sampler")
    if settings.get("sampler_seed_offset") != 1_000_000 or settings.get("initial_noise_device") != "cpu" or settings.get("batch_size_images") != 1:
        raise ValueError("Unsupported input noise or batch settings")
    steps = settings["sampling_steps"]
    if isinstance(steps, bool) or not isinstance(steps, int) or not 2 <= steps <= 1000:
        raise ValueError("sampling_steps must be an integer between 2 and 1000")
    if not isinstance(settings["cfg_scale"], (float, int)) or not 0 <= settings["cfg_scale"] < float("inf"):
        raise ValueError("cfg_scale must be finite and nonnegative")
    if not settings.get("vae_revision") or not settings.get("dit_commit") or not settings.get("checkpoint_sha256"):
        raise ValueError("Pin source, checkpoint and VAE revisions")
    return protocol


def protocol_hash(protocol):
    return hashlib.sha256(json.dumps(protocol, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def initial_latent(seed, device):
    """Same CPU FP32 initial noise on all machines; transfer only after generation."""
    return torch.randn((1, 4, 32, 32), dtype=torch.float32,
                       generator=torch.Generator(device="cpu").manual_seed(int(seed))).to(device)


def preprocess_image(image, expected_size=None):
    """Canonical LPIPS/storage input: one RGB FP32 CPU image in [-1, 1].

    Generated images are already 256x256: do not resize, crop or normalize each
    image independently. PNGs are previews; evaluation uses the saved tensors.
    """
    if not isinstance(image, torch.Tensor) or image.ndim != 4 or image.shape[:2] != (1, 3):
        raise ValueError("Expected image shape [1, 3, H, W]")
    if expected_size is not None and tuple(image.shape[-2:]) != (expected_size, expected_size):
        raise ValueError("Image size differs from the shared protocol")
    if not image.is_floating_point() or not torch.isfinite(image).all():
        raise ValueError("Images must contain finite floating-point values")
    if image.min() < -1.00001 or image.max() > 1.00001:
        raise ValueError("Expected decoded image range [-1, 1]")
    return image.detach().to(device="cpu", dtype=torch.float32).clamp(-1, 1).contiguous()


### 2.2. DEFINE — shared block schedules

In [22]:
"""Method-specific block schedules. Team members can add model transformations here."""
from torch import nn


class ScheduledBlocks(nn.Module):
    """Register each retained block ONCE; execute it at one or more depth slots.

    Official DiT.forward iterates over model.blocks. This container preserves that
    interface, while its state_dict contains only retained modules.
    """
    def __init__(self, blocks, removed=(), mode="original", alpha="inverse_k"):
        super().__init__()
        blocks = list(blocks)
        self.depth = len(blocks)
        self.mode = mode
        self.alpha = "inverse_k" if alpha == "inverse_k" else float(alpha)
        self.removed = sorted(set(int(i) for i in removed))
        if mode not in {"original", "removal", "loop", "damped"}:
            raise ValueError(f"Unknown mode: {mode}")
        if self.alpha != "inverse_k" and not 0 <= self.alpha <= 1:
            raise ValueError("Damping alpha must be inverse_k or a number in [0, 1].")
        if any(i < 0 or i >= self.depth for i in self.removed):
            raise ValueError("Removed index outside block range.")
        if mode == "original" and self.removed:
            raise ValueError("Original model cannot have removed blocks.")
        if 0 in self.removed:
            raise ValueError("Keep block 0: a replacement needs a preceding retained block.")
        self.retained = nn.ModuleDict({str(i): b for i, b in enumerate(blocks)
                                       if i not in self.removed})
        self.schedule = []
        previous = None
        iteration = 0
        for i in range(self.depth):
            if i not in self.removed:
                previous = i
                iteration = 1
                self.schedule.append((i, 1.0, False))
            elif mode != "removal":
                iteration += 1
                scale = (1.0 / iteration if self.alpha == "inverse_k" else self.alpha) if mode == "damped" else 1.0
                self.schedule.append((previous, scale, True))

    def __len__(self):
        return len(self.schedule)

    @staticmethod
    def _damped(block, scale):
        def call(x, c):
            if scale == 0:
                return x
            return x + scale * (block(x, c) - x)
        return call

    def __iter__(self):
        for index, scale, replacement in self.schedule:
            block = self.retained[str(index)]
            yield self._damped(block, scale) if replacement and scale != 1 else block

    def metadata(self):
        return dict(mode=self.mode, removed=self.removed, alpha=self.alpha,
                    damping_policy=("inverse_k" if self.alpha == "inverse_k" else "fixed") if self.mode == "damped" else "none",
                    depth=self.depth, retained=list(map(int, self.retained.keys())),
                    execution_schedule=[dict(source=i, scale=a, replacement=r)
                                        for i, a, r in self.schedule])


### 2.3. DEFINE — runtime and reproducibility utilities

In [23]:
"""Shared input, sampling, profiling and evaluation for CPU, MPS and CUDA.

Both notebooks import this implementation. Keep method changes in methods.py
or supply model_transform to run_variant.
"""

import gc
from contextlib import contextmanager
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
import time

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import torch
from torch import nn




def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")


def configure_dit(repo):
    repo = Path(repo).resolve()
    if not (repo / "models.py").exists():
        raise FileNotFoundError(f"Official DiT checkout missing: {repo}")
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    # Avoid silently using an unrelated notebook module called 'models'.
    import models
    if Path(models.__file__).resolve() != repo / "models.py":
        raise RuntimeError("An unrelated 'models' module is loaded. Restart the kernel.")
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=select_device().type == "mps")
    install_mps_sampler_compatibility()
    return subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()


def environment(requested="auto"):
    device = select_device(requested)
    packages = {name: importlib.metadata.version(name) for name in
                ["torch", "torchvision", "timm", "diffusers", "accelerate", "lpips", "numpy"]}
    return dict(python=sys.version, platform=platform.platform(), packages=packages,
                device=str(device), machine=platform.machine(), cuda=torch.version.cuda,
                gpu=torch.cuda.get_device_name(device) if device.type == "cuda" else None,
                mps_available=torch.backends.mps.is_available(), macos=platform.mac_ver()[0],
                mps_fallback=os.environ.get("PYTORCH_ENABLE_MPS_FALLBACK"),
                deterministic_warn_only=torch.is_deterministic_algorithms_warn_only_enabled())


def select_device(requested="auto"):
    device = str(requested)
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    result = torch.device(device)
    if result.type not in {"cpu", "mps", "cuda"}:
        raise ValueError("device must be auto, cpu, mps or cuda[:index]")
    if result.type == "mps" and not torch.backends.mps.is_available():
        raise RuntimeError("MPS unavailable in this Python process.")
    if result.type == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA unavailable: select a GPU machine and CUDA-enabled PyTorch.")
        if result.index is None:
            result = torch.device("cuda", torch.cuda.current_device())
        if result.index >= torch.cuda.device_count():
            raise ValueError("CUDA device index outside available range")
    return result


def install_mps_sampler_compatibility():
    from diffusion import gaussian_diffusion as gd
    if getattr(gd._extract_into_tensor, "_mac_compatible", False):
        return
    original = gd._extract_into_tensor
    def extract(arr, timesteps, broadcast_shape):
        if timesteps.device.type != "mps":
            return original(arr, timesteps, broadcast_shape)
        # Same selected coefficients, cast BEFORE device transfer (MPS has no float64).
        values = torch.from_numpy(arr).float().to(timesteps.device)[timesteps]
        while values.ndim < len(broadcast_shape):
            values = values[..., None]
        return values.expand(broadcast_shape)
    extract._mac_compatible = True
    gd._extract_into_tensor = extract


def synchronize(device):
    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize(device)


@contextmanager
def paired_rng(device, seed):
    """Restore CPU and selected accelerator RNG, including after exceptions."""
    state = torch.mps.get_rng_state() if device.type == "mps" else None
    devices = [device.index if device.index is not None else torch.cuda.current_device()] if device.type == "cuda" else []
    with torch.random.fork_rng(devices=devices):
        try:
            torch.random.default_generator.manual_seed(int(seed))
            if device.type == "mps":
                torch.mps.manual_seed(int(seed))
            elif device.type == "cuda":
                with torch.cuda.device(device):
                    torch.cuda.manual_seed(int(seed))
            yield
        finally:
            if state is not None:
                torch.mps.set_rng_state(state)


### 2.4. DEFINE — feature profiling and block selection

In [24]:
class FeatureProfiler:
    """Per-sample L1 ratios, averaged separately over CFG's two branches.

    Hooks run only on original-model calibration trajectories. Scalar tensors
    stay on the device until collection ends, avoiding per-block CPU synchronization.
    """
    def __init__(self, model):
        self.model = model
        self.handles = []
        self.records = []
        self.timestep = None

    def __enter__(self):
        if self.model.blocks.mode != "original":
            raise ValueError("Profile original model only.")
        # Official forward_with_cfg calls self.forward directly, bypassing root
        # Module hooks. The timestep embedder is still invoked through __call__.
        self.handles.append(self.model.t_embedder.register_forward_pre_hook(self._before))
        for index, block in self.model.blocks.retained.items():
            def hook(module, args, output, index=int(index)):
                x = args[0].float()
                ratio = (output.float() - x).abs().flatten(1).sum(1) / (
                    x.abs().flatten(1).sum(1) + 1e-8)
                cond, uncond = ratio.chunk(2)
                self.records.append(torch.stack([
                    self.timestep.float(), ratio.new_tensor(index), cond.mean(), uncond.mean()
                ]).detach())
            self.handles.append(block.register_forward_hook(hook))
        return self

    def _before(self, module, args):
        self.timestep = args[0][0].detach()

    def __exit__(self, *exc):
        for handle in self.handles:
            handle.remove()
        self.handles.clear()

    def frame(self):
        import pandas as pd
        if not self.records:
            raise RuntimeError("No profile records collected.")
        values = torch.stack(self.records).cpu().numpy()
        frame = pd.DataFrame(values, columns=["timestep", "block", "conditional", "unconditional"])
        frame[["timestep", "block"]] = frame[["timestep", "block"]].astype(int)
        frame["mean_cfg"] = frame[["conditional", "unconditional"]].mean(axis=1)
        return frame


def summarize_profile(frame, output, remove_count=4):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import pandas as pd
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    # Average samples within each timestep first, then give each timestep equal weight.
    matrix = frame.groupby(["block", "timestep"])["mean_cfg"].mean().unstack()
    matrix = matrix.reindex(columns=sorted(matrix.columns, reverse=True))
    if matrix.isna().any().any() or not np.isfinite(matrix.to_numpy()).all():
        raise ValueError("Incomplete or nonfinite feature profile.")
    scores = matrix.mean(axis=1).rename("mean_feature_change").to_frame()
    depth = len(scores)
    candidates = scores.loc[~scores.index.isin([0, depth - 1])]
    if not 0 < remove_count <= len(candidates):
        raise ValueError("Invalid removal budget.")
    removed = sorted(candidates.sort_values("mean_feature_change", kind="stable").head(remove_count).index.tolist())
    scores["selected"] = scores.index.isin(removed)
    # Stages refer to sampling order, NOT increasing numerical diffusion t.
    for stage, columns in zip(["early_noisy", "middle", "late_clean"],
                              np.array_split(matrix.columns.to_numpy(), 3)):
        scores[stage] = matrix.loc[:, columns].mean(axis=1)
    scores.to_csv(output / "block_scores.csv", index_label="block")
    matrix.to_csv(output / "feature_change_matrix.csv")
    write_json(output / "selection.json", dict(removed=removed, protected=[0, depth - 1],
        scoring="mean of per-sample normalized L1 changes, equally averaged over CFG branches and timesteps"))
    fig, axes = plt.subplots(1, 2, figsize=(15, 7), constrained_layout=True)
    for ax, values, title in zip(axes, [matrix.to_numpy(), np.log10(matrix.to_numpy().clip(1e-12))],
                                  ["Mean normalized feature change", "log10 normalized feature change"]):
        im = ax.imshow(values, aspect="auto", interpolation="nearest")
        ticks = np.unique(np.linspace(0, len(matrix.columns) - 1, min(6, len(matrix.columns))).astype(int))
        ax.set_xticks(ticks, [str(matrix.columns[i]) for i in ticks])
        ax.set_yticks(range(depth))
        ax.set_xlabel("Diffusion timestep (sampling proceeds left to right)")
        ax.set_ylabel("Block index (zero based)")
        ax.set_title(title)
        for i in removed:
            ax.axhline(i, color="red", linewidth=0.6, alpha=0.6)
        fig.colorbar(im, ax=ax)
    fig.savefig(output / "feature_change_heatmap.png", dpi=180)
    plt.close(fig)
    return removed, scores


### 2.5. DEFINE — model loading and sampling

In [25]:
def load_model(checkpoint, removed=(), mode="original", alpha="inverse_k"):
    from models import DiT_XL_2
    # Avoid allocating a second full set of initialized XL weights in unified memory.
    with torch.device("meta"):
        model = DiT_XL_2(input_size=32, num_classes=1000)
    state = torch.load(checkpoint, map_location="cpu", weights_only=True, mmap=True)
    model.load_state_dict(state, strict=True, assign=True)
    del state
    model.blocks = ScheduledBlocks(model.blocks, removed, mode, alpha)
    return model.eval().requires_grad_(False)


def load_compact_checkpoint(path):
    """Rebuild the schedule before loading; plain official DiT cannot load this format."""
    from models import DiT_XL_2
    payload = torch.load(path, map_location="cpu", weights_only=True, mmap=True)
    meta = payload["schedule"]
    if meta.get("mode") == "custom":
        raise ValueError("Custom checkpoints require the corresponding method builder before loading state_dict")
    # Avoid allocating a second full set of initialized XL weights in unified memory.
    with torch.device("meta"):
        model = DiT_XL_2(input_size=32, num_classes=1000)
    model.blocks = ScheduledBlocks(model.blocks, meta["removed"], meta["mode"], meta["alpha"])
    model.load_state_dict(payload["state_dict"], strict=True, assign=True)
    return model.eval().requires_grad_(False)


def checkpoint_stats(model, path, keep=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    # CPU serialization happens before the model goes to device; no hidden device copies.
    torch.save(dict(format_version=1, architecture="DiT-XL/2-256",
                    schedule=(model.blocks.metadata() if hasattr(model.blocks, "metadata") else dict(mode="custom")), state_dict=model.state_dict()), path)
    result = dict(unique_parameters=sum(p.numel() for p in model.parameters()),
                  checkpoint_bytes=path.stat().st_size,
                  parameter_bytes=sum(p.numel() * p.element_size() for p in model.parameters()),
                  unique_blocks=(len(model.blocks.retained) if hasattr(model.blocks, "retained") else len(model.blocks)),
                  block_evaluations_per_forward=len(model.blocks))
    if not keep:
        path.unlink()
    return result


def download_checkpoint(path):
    path = Path(path)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.hub.download_url_to_file(
            "https://dl.fbaipublicfiles.com/DiT/models/DiT-XL-2-256x256.pt", str(path))
    return path


@torch.inference_mode()
def generate(model, vae, diffusion, label, seed, cfg_scale=4.0, progress=False):
    device = next(model.parameters()).device
    # Independent CPU generator makes initial latents independent of prior work.
    z = initial_latent(seed, device)
    z = torch.cat([z, z], dim=0)
    labels = torch.tensor([int(label), 1000], device=device)
    # Reset the complete ancestral-sampler noise stream for every paired sample.
    with paired_rng(device, int(seed) + 1_000_000):
        samples = diffusion.p_sample_loop(model.forward_with_cfg, z.shape, noise=z,
            clip_denoised=False, model_kwargs=dict(y=labels, cfg_scale=cfg_scale),
            progress=progress, device=device)
        image = vae.decode(samples[:1] / 0.18215).sample.clamp(-1, 1)
    if not torch.isfinite(image).all():
        raise RuntimeError(f"Nonfinite image: label={label}, seed={seed}")
    return preprocess_image(image)


@torch.inference_mode()
def warmup(model, vae, cfg_scale):
    device = next(model.parameters()).device
    z = torch.zeros(2, 4, 32, 32, device=device)
    model.forward_with_cfg(z, torch.full((2,), 999, device=device, dtype=torch.long),
                           torch.tensor([0, 1000], device=device), cfg_scale)
    vae.decode(z[:1] / 0.18215)
    synchronize(device)


def _prepare(config, mode, removed, alpha, model_transform=None):
    device = select_device(config.get("device", "auto"))
    torch.use_deterministic_algorithms(True, warn_only=device.type == "mps")
    from diffusion import create_diffusion
    from diffusers import AutoencoderKL
    model = load_model(config["checkpoint"], removed, mode, alpha)
    if model_transform is not None:
        model = model_transform(model)
        if not isinstance(model, nn.Module):
            raise TypeError("model_transform must return a torch.nn.Module")
        model.eval().requires_grad_(False)
    vae = AutoencoderKL.from_pretrained(config["vae_id"], revision=config.get("vae_revision"))
    vae.eval().requires_grad_(False)
    diffusion = create_diffusion(str(config["sampling_steps"]))
    return model, vae, diffusion


def _cleanup():
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


### 2.6. DEFINE — calibration and variant execution

In [26]:
def calibrate(config, pairs, output):
    import pandas as pd
    pairs = validate_pairs(pairs)
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    model, vae, diffusion = _prepare(config, "original", (), 1)
    try:
        device = select_device(config.get("device", "auto"))
        model.to(device); vae.to(device)
        frames = []
        for label, seed in pairs:
            with FeatureProfiler(model) as profiler:
                generate(model, vae, diffusion, label, seed, config["cfg_scale"])
            frame = profiler.frame()
            frame["label"], frame["seed"] = label, seed
            frames.append(frame)
            print(f"Profiled label={label}, seed={seed}", flush=True)
            del profiler
        frame = pd.concat(frames, ignore_index=True)
        frame.to_csv(output / "feature_changes.csv", index=False)
        write_json(output / "calibration_manifest.json", dict(config=config, pairs=pairs))
        return frame
    finally:
        del model, vae, diffusion
        _cleanup()


def run_variant(config, pairs, output, mode="original", removed=(), alpha="inverse_k",
                keep_checkpoint=False, model_transform=None, method_metadata=None):
    import pandas as pd
    from torchvision.utils import save_image
    pairs = validate_pairs(pairs)
    select_device(config.get("device", "auto"))
    if model_transform is not None and (mode != "original" or removed):
        raise ValueError("Use either a custom transform or a built-in mode, not both")
    output = Path(output)
    # Never silently mix outputs from different settings or overwrite prior results.
    output.mkdir(parents=True, exist_ok=False)
    model, vae, diffusion = _prepare(config, mode, removed, alpha, model_transform)
    try:
        schedule = model.blocks.metadata() if hasattr(model.blocks, "metadata") else dict(mode="custom")
        stats = checkpoint_stats(model, output / "model.pt", keep_checkpoint)
        write_json(output / "manifest.json", dict(config=config, pairs=pairs,
                   schedule=schedule, stats=stats, method_metadata=method_metadata or {},
                   shared_code_sha256=shared_code_hash(),
                   runtime=environment(config.get("device", "auto")),
                   vae_commit=getattr(vae.config, "_commit_hash", None),
                   timing="synchronized diffusion + VAE decode + CPU image transfer; excludes loading, warmup, PNG/checkpoint writes, profiling and LPIPS"))
        device = select_device(config.get("device", "auto"))
        model.to(device); vae.to(device)
        warmup(model, vae, config["cfg_scale"])
        rows, images = [], []
        for index, (label, seed) in enumerate(pairs):
            _cleanup()
            synchronize(device)
            if device.type == "cuda":
                torch.cuda.reset_peak_memory_stats(device)
            start = time.perf_counter()
            image = generate(model, vae, diffusion, label, seed, config["cfg_scale"])
            synchronize(device)
            elapsed = time.perf_counter() - start
            row = dict(index=index, label=label, seed=seed, latency_s=elapsed,
                       device=str(device),
                       peak_allocated_bytes=(torch.cuda.max_memory_allocated(device) if device.type == "cuda" else None),
                       peak_reserved_bytes=(torch.cuda.max_memory_reserved(device) if device.type == "cuda" else None),
                       mps_allocated_bytes_after_sample=(torch.mps.current_allocated_memory()
                           if device.type == "mps" else None),
                       mps_driver_bytes_after_sample=(torch.mps.driver_allocated_memory()
                           if device.type == "mps" else None))
            rows.append(row)
            # Keep only a small preview in RAM, even for 5,000-image runs.
            if len(images) < 16:
                images.append(image)
            image = preprocess_image(image, config["image_size"])
            torch.save(image, output / f"image_{index:04d}.pt")
            save_image(image, output / f"image_{index:04d}.png", normalize=True, value_range=(-1, 1))
            pd.DataFrame(rows).to_csv(output / "samples.csv", index=False)
            print(f"{mode}: {index + 1}/{len(pairs)}, {elapsed:.2f} s", flush=True)
        save_image(torch.cat(images), output / "grid.png", nrow=4,
                   normalize=True, value_range=(-1, 1))
        return pd.DataFrame(rows)
    finally:
        del model, vae, diffusion
        _cleanup()


### 2.7. DEFINE — metrics, comparison grids and file hashing

In [27]:
@torch.inference_mode()
def compare_results(root, names, metric=None):
    """LPIPS runs on CPU after all device models are released; never contaminates timing."""
    import pandas as pd
    root = Path(root)
    if metric is None:
        import lpips
        metric = lpips.LPIPS(net="alex", version="0.1").cpu().eval()
    names = list(names)
    if not names or len(set(names)) != len(names):
        raise ValueError("Provide nonempty, unique variant names")
    baseline = pd.read_csv(root / "original" / "samples.csv")
    ref_manifest = json.loads((root / "original" / "manifest.json").read_text())
    all_rows, summary = [], []
    for name in names:
        folder = root / name
        samples = pd.read_csv(folder / "samples.csv")
        manifest = json.loads((folder / "manifest.json").read_text())
        if manifest.get("shared_code_sha256") != ref_manifest.get("shared_code_sha256"):
            raise ValueError(f"Shared pipeline code differs: {name}")
        if manifest.get("runtime") != ref_manifest.get("runtime"):
            raise ValueError(f"Hardware or software environment differs: {name}")
        if manifest["config"] != ref_manifest["config"] or manifest["pairs"] != ref_manifest["pairs"]:
            raise ValueError(f"Mismatched configuration or sample pairs: {name}")
        if not samples[["index", "label", "seed"]].equals(baseline[["index", "label", "seed"]]):
            raise ValueError(f"Mismatched sample order: {name}")
        pairs = validate_pairs(manifest["pairs"])
        expected = pd.DataFrame([(i, label, seed) for i, (label, seed) in enumerate(pairs)],
                                columns=["index", "label", "seed"])
        if not samples[["index", "label", "seed"]].equals(expected):
            raise ValueError(f"Incomplete or invalid sample manifest: {name}")
        scores = []
        for i in samples["index"]:
            ref = torch.load(root / "original" / f"image_{i:04d}.pt", weights_only=True)
            image = torch.load(folder / f"image_{i:04d}.pt", weights_only=True)
            ref = preprocess_image(ref, manifest["config"]["image_size"])
            image = preprocess_image(image, manifest["config"]["image_size"])
            score = float(metric(ref, image).item())
            if not np.isfinite(score):
                raise ValueError("Nonfinite paired metric")
            scores.append(score)
        samples["paired_lpips"] = scores
        samples["variant"] = name
        all_rows.append(samples)
        stats = manifest["stats"]
        summary.append(dict(variant=name, images=len(samples), lpips_mean=float(np.mean(scores)),
            lpips_std=float(np.std(scores, ddof=1)) if len(scores) > 1 else 0.0,
            unique_parameters=stats["unique_parameters"], unique_blocks=stats["unique_blocks"],
            block_evaluations_per_forward=stats["block_evaluations_per_forward"],
            checkpoint_MiB=stats["checkpoint_bytes"] / 2**20,
            device=samples["device"].iloc[0] if "device" in samples else "cuda",
            peak_GPU_allocated_GiB=(samples["peak_allocated_bytes"].max() / 2**30 if "peak_allocated_bytes" in samples else None),
            peak_GPU_reserved_GiB=(samples["peak_reserved_bytes"].max() / 2**30 if "peak_reserved_bytes" in samples else None),
            max_post_sample_MPS_allocated_GiB=(samples["mps_allocated_bytes_after_sample"].max() / 2**30 if "mps_allocated_bytes_after_sample" in samples else None),
            max_post_sample_MPS_driver_GiB=(samples["mps_driver_bytes_after_sample"].max() / 2**30 if "mps_driver_bytes_after_sample" in samples else None),
            latency_mean_s=samples["latency_s"].mean(), latency_std_s=samples["latency_s"].std(ddof=0)))
    result = pd.DataFrame(summary)
    result.to_csv(root / "results.csv", index=False)
    pd.concat(all_rows, ignore_index=True).to_csv(root / "paired_metrics.csv", index=False)
    write_json(root / "evaluation.json", dict(
        metric="lpips_alex_v0.1" if metric.__class__.__module__.startswith("lpips") else "injected_metric",
        preprocessing="saved CPU FP32 RGB tensors in [-1,1]; no resizing",
        variants=names, images=len(baseline), paired_reference="original",
        shared_code_sha256=shared_code_hash(),
        interpretation="distance from original output, not absolute generation quality"))
    return result


def comparison_grid(root, names, pairs, count=8):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from PIL import Image
    root = Path(root)
    count = min(count, len(pairs))
    fig, axes = plt.subplots(len(names), count, figsize=(2.4 * count, 2.7 * len(names)),
                             squeeze=False, constrained_layout=True)
    for row, name in enumerate(names):
        for column in range(count):
            ax = axes[row, column]
            with Image.open(root / name / f"image_{column:04d}.png") as image:
                ax.imshow(image)
            ax.set_xticks([]); ax.set_yticks([])
            if row == 0:
                ax.set_title(f"class {pairs[column][0]}\nseed {pairs[column][1]}", fontsize=9)
            if column == 0:
                ax.set_ylabel(name)
    fig.savefig(root / "comparison_grid.png", dpi=160)
    plt.close(fig)


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


### 2.8. DEFINE — notebook source fingerprints and run preparation

In [28]:
import inspect
from types import SimpleNamespace

SHARED_DEFINITION_NAMES = ['validate_pairs', 'make_protocol', 'load_protocol', 'protocol_hash', 'initial_latent', 'preprocess_image', 'ScheduledBlocks', 'write_json', 'configure_dit', 'environment', 'select_device', 'install_mps_sampler_compatibility', 'synchronize', 'paired_rng', 'FeatureProfiler', 'summarize_profile', 'load_model', 'load_compact_checkpoint', 'checkpoint_stats', 'download_checkpoint', 'generate', 'warmup', '_prepare', '_cleanup', 'calibrate', 'run_variant', 'compare_results', 'comparison_grid', 'sha256']

def notebook_sources():
    """Read executed function sources from IPython's source cache."""
    sources = {}
    for name in SHARED_DEFINITION_NAMES:
        obj = globals()[name]
        if inspect.isclass(obj):
            # Inspect methods individually: __main__ classes lack a module file.
            for member_name, member in vars(obj).items():
                if isinstance(member, (staticmethod, classmethod)):
                    member = member.__func__
                if inspect.isfunction(member):
                    sources[f"{name}.{member_name}"] = inspect.getsource(member)
        else:
            sources[name] = inspect.getsource(obj)
    return sources

def shared_code_hash():
    return hashlib.sha256(json.dumps(notebook_sources(), sort_keys=True).encode()).hexdigest()

def prepare(args):
    protocol = load_protocol(args.protocol)
    device = select_device(args.device)
    commit = configure_dit(args.repo)
    if commit != protocol["settings"]["dit_commit"]:
        raise ValueError("DiT source commit differs from the agreed protocol")
    checkpoint = download_checkpoint(Path(args.checkpoint).resolve())
    if sha256(checkpoint) != protocol["settings"]["checkpoint_sha256"]:
        raise ValueError("Checkpoint hash differs from the agreed protocol")
    root = Path(args.output)
    root.mkdir(parents=True, exist_ok=False)
    config = {**protocol["settings"], "checkpoint": str(checkpoint), "device": str(device),
              "protocol_sha256": protocol_hash(protocol), "shared_code_sha256": shared_code_hash()}
    write_json(root / "protocol.json", protocol)
    write_json(root / "notebook_sources.json", notebook_sources())
    write_json(root / "config.json", config)
    write_json(root / "environment.json", environment(str(device)))
    with (root / "pip-freeze.txt").open("w") as out:
        subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=out, check=True)
    print(f"Prepared shared inputs: {root.resolve()}")

def load_run(args):
    root = Path(args.output)
    protocol = load_protocol(root / "protocol.json")
    config = json.loads((root / "config.json").read_text())
    if config.get("shared_code_sha256") != shared_code_hash():
        raise ValueError("Shared pipeline changed after preparation; create a new run")
    if config["protocol_sha256"] != protocol_hash(protocol):
        raise ValueError("Protocol changed after preparation; create a new run")
    if any(config.get(k) != v for k, v in protocol["settings"].items()):
        raise ValueError("Config settings changed after preparation; create a new run")
    if configure_dit(args.repo) != config["dit_commit"]:
        raise ValueError("DiT source changed after preparation")
    if sha256(config["checkpoint"]) != config["checkpoint_sha256"]:
        raise ValueError("Checkpoint changed after preparation")
    current = environment(config["device"])
    recorded = json.loads((root / "environment.json").read_text())
    if current != recorded:
        raise ValueError("Runtime environment changed; prepare a new run for comparable measurements")
    return root, protocol, config


### 2.9. DEFINE — Imagenette class array, sample counts, custom method and runners

In [29]:
# Imagenette v2: original ImageNet-1K ids, synsets and names (NOT local ids 0..9).
# Source: https://github.com/fastai/imagenette
IMAGENETTE_CLASSES = (
    (0,   "n01440764", "tench"),
    (217, "n02102040", "English springer"),
    (482, "n02979186", "cassette player"),
    (491, "n03000684", "chain saw"),
    (497, "n03028079", "church"),
    (566, "n03394916", "French horn"),
    (569, "n03417042", "garbage truck"),
    (571, "n03425413", "gas pump"),
    (574, "n03445777", "golf ball"),
    (701, "n03888257", "parachute"),
)
# The single class array used by every generation method and split.
# A subset of these ids is also supported by the reference preparation below.
CLASS_LABELS = tuple(label for label, synset, name in IMAGENETTE_CLASSES)
CALIBRATION_IMAGES_PER_CLASS = 2
VALIDATION_IMAGES_PER_CLASS = 2  # 20 images/method for selecting settings.
FINAL_IMAGES_PER_CLASS = 500   # 5,000 images/method in the held-out test.
FID_REAL_IMAGES_PER_CLASS = 500  # Balanced, fixed real reference: 5,000 images.

PROTOCOL = make_protocol(
    labels=CLASS_LABELS,
    calibration_per_class=CALIBRATION_IMAGES_PER_CLASS,
    validation_per_class=VALIDATION_IMAGES_PER_CLASS,
    test_per_class=FINAL_IMAGES_PER_CLASS,
)
CUSTOM_SETTINGS = {"removed": [4], "mode": "damped", "alpha": "inverse_k"}

"""Copy this file for each member's method; leave the shared pipeline unchanged."""


def build_model(model, settings):
    """Receive the same pretrained CPU DiT and return the modified model.

    settings is a JSON object supplied with --method-config. Use calibration and
    validation data to choose it, then freeze it for the held-out test split.
    A future training method can train here and return the resulting model; the
    runner always evaluates in inference mode with gradients disabled afterward.
    """
    model.blocks = ScheduledBlocks(
        model.blocks.retained.values(),
        removed=settings["removed"],
        mode=settings.get("mode", "damped"),
        alpha=settings.get("alpha", "inverse_k"),
    )
    return model


def run_method(split, method, name, *, alpha="inverse_k", builder=None, settings=None):
    """Direct notebook entry point; keeps the CLI's run checks."""
    if split not in {"validation", "test"}:
        raise ValueError("Use validation or test")
    if Path(name).name != name or name in {".", "..", ""}:
        raise ValueError("name must be a single folder name")
    if (name == "original") != (method == "original" and builder is None):
        raise ValueError("Reserve original for the unmodified baseline")
    root, protocol, config = load_run(RUN_ARGS)
    removed, transform = (), None
    metadata = dict(method=method)
    if builder is not None:
        settings = json.loads(json.dumps(settings or {}))
        source = inspect.getsource(builder)
        metadata.update(settings=settings, source=source,
                        source_sha256=hashlib.sha256(source.encode()).hexdigest())
        transform = lambda model: builder(model, settings)
        mode = "original"
    elif method in {"removal", "loop", "damped"}:
        removed = json.loads((root / "profile/selection.json").read_text())["removed"]
        mode = method
    elif method == "original":
        mode = method
    else:
        raise ValueError("Unknown method; pass builder= for a custom method")
    return run_variant(config, protocol["splits"][split], root / split / name,
                       mode=mode, removed=removed, alpha=alpha, model_transform=transform,
                       method_metadata=metadata, keep_checkpoint=KEEP_CHECKPOINTS)


def evaluate_split(split, names):
    root, protocol, config = load_run(RUN_ARGS)
    result = compare_results(root / split, names)
    comparison_grid(root / split, names, protocol["splits"][split])
    return result


### 2.10. DEFINE — CPU tests from tests_team

In [30]:
"""Shared protocol and end-to-end artifact checks; no pretrained downloads."""
import copy
import json
from types import SimpleNamespace

import pandas as pd
import pytest
import torch
from torch import nn



def test_protocol_has_balanced_disjoint_splits_and_stable_hash(tmp_path):
    protocol = make_protocol(labels=[1, 2], calibration_per_class=2,
                             validation_per_class=1, test_per_class=3)
    path = tmp_path / "protocol.json"
    path.write_text(json.dumps(protocol))
    assert load_protocol(path) == protocol
    assert [len(v) for v in protocol["splits"].values()] == [4, 2, 6]
    reordered = dict(reversed(list(protocol.items())))
    assert protocol_hash(protocol) == protocol_hash(reordered)
    changed = copy.deepcopy(protocol)
    changed["splits"]["test"][0][1] = changed["splits"]["calibration"][0][1]
    path.write_text(json.dumps(changed))
    with pytest.raises(ValueError, match="leakage"):
        load_protocol(path)


def test_input_validation_rejects_ambiguous_samples(pairs):
    with pytest.raises(ValueError):
        validate_pairs(pairs)


def test_preprocessing_and_initial_noise_are_fixed():
    before = torch.get_rng_state().clone()
    a = initial_latent(20, "cpu")
    torch.randn(5)
    assert torch.equal(a, initial_latent(20, "cpu"))
    torch.set_rng_state(before)
    x = torch.zeros(1, 3, 8, 8, dtype=torch.float64, requires_grad=True)
    y = preprocess_image(x, 8)
    assert y.dtype == torch.float32 and y.device.type == "cpu" and not y.requires_grad
    for invalid in [torch.full_like(x, float("nan")), x + 2, x.to(torch.uint8), x.squeeze(0)]:
        with pytest.raises(ValueError):
            preprocess_image(invalid, 8)
    with pytest.raises(ValueError, match="size"):
        preprocess_image(x, 16)


class Tiny(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = ScheduledBlocks([nn.Linear(1, 1), nn.Linear(1, 1)])


def test_shared_runner_and_evaluator_reject_incomplete_or_mismatched_runs(tmp_path, monkeypatch):
    # The actual save/load, manifest, timing and evaluation code runs unchanged.
    vae = nn.Linear(1, 1)
    vae.config = SimpleNamespace()
    monkeypatch.setitem(globals(), "_prepare", lambda *a: (Tiny(), vae, None))
    monkeypatch.setitem(globals(), "warmup", lambda *a: None)
    monkeypatch.setitem(globals(), "generate", lambda model, vae, diffusion, label, seed, cfg:
                        torch.full((1, 3, 8, 8), seed / 100))
    config = dict(device="cpu", image_size=8, cfg_scale=4)
    pairs = [(1, 10), (2, 20)]
    for name in ["original", "member1"]:
        run_variant(config, pairs, tmp_path / name)
    metric = lambda x, y: (x - y).square().mean()
    result = compare_results(tmp_path, ["original", "member1"], metric=metric)
    assert result.lpips_mean.eq(0).all()
    assert result.images.eq(2).all()
    assert result.peak_GPU_allocated_GiB.isna().all()
    assert (tmp_path / "paired_metrics.csv").exists()
    with pytest.raises(FileExistsError):
        run_variant(config, pairs, tmp_path / "member1")
    samples_path = tmp_path / "member1/samples.csv"
    samples = pd.read_csv(samples_path)
    samples.iloc[:1].to_csv(samples_path, index=False)
    with pytest.raises(ValueError, match="sample order"):
        compare_results(tmp_path, ["member1"], metric=metric)
    samples.to_csv(samples_path, index=False)
    path = tmp_path / "member1/manifest.json"
    manifest = json.loads(path.read_text())
    manifest["config"]["cfg_scale"] = 5
    path.write_text(json.dumps(manifest))
    with pytest.raises(ValueError, match="configuration"):
        compare_results(tmp_path, ["member1"], metric=metric)


def test_custom_builder_keeps_shared_preparation_and_freezes_model(monkeypatch):
    import sys
    import types
    monkeypatch.setitem(globals(), "load_model", lambda *a: Tiny())
    vae = nn.Linear(1, 1)
    factory = types.SimpleNamespace(from_pretrained=lambda *a, **k: vae)
    monkeypatch.setitem(sys.modules, "diffusers", types.SimpleNamespace(AutoencoderKL=factory))
    monkeypatch.setitem(sys.modules, "diffusion", types.SimpleNamespace(create_diffusion=lambda *a: "sampler"))
    seen = []
    def builder(model):
        seen.append(model)
        model.blocks = ScheduledBlocks(model.blocks.retained.values(), [1], "damped", .25)
        return model
    model, loaded_vae, diffusion = _prepare(dict(device="cpu", checkpoint="unused",
        vae_id="unused", sampling_steps=50), "original", (), .5, builder)
    assert seen == [model] and not model.training
    assert all(not p.requires_grad for p in model.parameters())
    assert len(model.blocks.retained) == 1
    assert loaded_vae is vae and diffusion == "sampler"


In [31]:
# XSA definitions — execute after the shared definitions and before tests/preparation.
from contextlib import contextmanager
import math
import json
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F


def xsa_filter_heads(output, value, eps=1e-8):
    """Remove each token/head's current self-value direction; last axis is head_dim."""
    if output.shape != value.shape or output.ndim != 4:
        raise ValueError("XSA needs matching four-dimensional head output/value tensors")
    if not math.isfinite(eps) or eps <= 0:
        raise ValueError("XSA epsilon must be finite and positive")
    # FP32 projection also makes eps safe if this helper is used with FP16/BF16.
    dtype = torch.float32 if output.dtype in {torch.float16, torch.bfloat16} else output.dtype
    o, v = output.to(dtype), value.to(dtype)
    v_normalized = F.normalize(v, dim=-1, eps=eps)
    z = o - (o * v_normalized).sum(dim=-1, keepdim=True) * v_normalized
    return z.to(output.dtype)


@contextmanager
def xsa_attention_scope(attention, enabled=True, eps=1e-8):
    """Filter exactly one ordinary timm Attention call, with no persistent flags.

    The original forward computes fresh Q/K/V and attention (including its fused
    or unfused path, Q/K norms, scaling and masks). At the head-concatenation
    boundary we view the tensor as separate heads, filter, and concatenate again.
    This is exactly the pre-concatenation projection: reshape mixes no values.
    timm's optional post-attention norm/gate and output projection run afterward.
    Hooks and this call's value tensor are always discarded, even on failure.
    Intended for this notebook's sequential eager inference, not concurrent calls
    to the same module or torch.compile.
    """
    if not enabled:
        yield
        return
    from timm.models.vision_transformer import Attention
    if type(attention) is not Attention:
        raise TypeError("XSA adapter supports the standard timm Attention used by official DiT")
    if not math.isfinite(eps) or eps <= 0:
        raise ValueError("XSA epsilon must be finite and positive")
    state = {}
    counts = {"qkv": 0, "filter": 0}

    def capture_value(module, args, qkv):
        if counts["qkv"]:
            raise RuntimeError("Expected exactly one fresh QKV evaluation per XSA call")
        b, n, three_c = qkv.shape
        h = attention.num_heads
        if three_c % (3 * h) or three_c // 3 != attention.proj.in_features:
            raise RuntimeError("Unsupported timm QKV/head layout")
        # Only THIS evaluation's V is retained until its attention output arrives.
        state["value"] = qkv.reshape(b, n, 3, h, three_c // (3 * h))[:, :, 2]
        counts["qkv"] += 1

    def filter_before_projection(module, args):
        if "value" not in state:
            raise RuntimeError("XSA output did not follow a fresh QKV evaluation")
        value = state.pop("value")
        output = args[0]
        if output.shape != (*value.shape[:2], value.shape[2] * value.shape[3]):
            raise RuntimeError("Unsupported timm attention-output layout")
        heads = output.reshape_as(value)
        filtered = xsa_filter_heads(heads, value, eps).reshape_as(output)
        counts["filter"] += 1
        return (filtered, *args[1:])

    handles = []
    try:
        handles.append(attention.qkv.register_forward_hook(capture_value))
        # New timm has a post-head-concatenation norm; old timm goes straight to
        # proj. Filtering BEFORE that norm preserves it, rather than bypassing it.
        boundary = getattr(attention, "norm", None)
        if boundary is None:
            boundary = attention.proj
        handles.append(boundary.register_forward_pre_hook(filter_before_projection))
        yield
        if counts != {"qkv": 1, "filter": 1}:
            raise RuntimeError("XSA hooks did not observe exactly one attention evaluation")
    finally:
        for handle in handles:
            handle.remove()
        state.clear()


class ScheduledBlocks(nn.Module):
    """Retain original indices and register each retained block exactly once."""
    def __init__(self, blocks, removed=(), mode="original", alpha="inverse_k", xsa_eps=1e-8):
        super().__init__()
        blocks = list(blocks)
        self.depth = len(blocks)
        self.mode = mode
        self.alpha = "inverse_k" if alpha == "inverse_k" else float(alpha)
        self.xsa_eps = float(xsa_eps)
        self.removed = sorted(set(int(i) for i in removed))
        if mode not in {"original", "removal", "loop", "damped", "xsa_reuse"}:
            raise ValueError(f"Unknown mode: {mode}")
        if self.alpha != "inverse_k" and not 0 <= self.alpha <= 1:
            raise ValueError("Damping alpha must be inverse_k or a number in [0, 1]")
        if not math.isfinite(self.xsa_eps) or self.xsa_eps <= 0:
            raise ValueError("XSA epsilon must be finite and positive")
        if any(i < 0 or i >= self.depth for i in self.removed):
            raise ValueError("Removed index outside block range")
        if mode == "original" and self.removed:
            raise ValueError("Original model cannot have removed blocks")
        if 0 in self.removed:
            raise ValueError("Keep block 0: a replacement needs a preceding retained block")
        self.retained = nn.ModuleDict({str(i): b for i, b in enumerate(blocks)
                                       if i not in self.removed})
        self.schedule = []
        self.slots = []
        self.iterations = []
        previous = None
        iteration = 0
        for slot in range(self.depth):
            if slot not in self.removed:
                previous = slot
                iteration = 1
                self.schedule.append((slot, 1.0, False))
                self.slots.append(slot)
                self.iterations.append(iteration)
            elif mode != "removal":
                # k includes the original call; each retained block starts at k=1.
                # Precompute per-slot values so k resets on every model forward.
                iteration += 1
                scale = (1.0 / iteration if self.alpha == "inverse_k" else self.alpha) if mode == "damped" else 1.0
                self.schedule.append((previous, scale, True))
                self.slots.append(slot)
                self.iterations.append(iteration)

    def __len__(self):
        return len(self.schedule)

    @staticmethod
    def _damped(block, scale):
        def call(x, c):
            if scale == 0:
                return x
            return x + scale * (block(x, c) - x)
        return call

    @staticmethod
    def _xsa(block, eps):
        def call(x, c):
            # Keep DiTBlock.forward itself: adaLN, both gates, residuals and MLP.
            with xsa_attention_scope(block.attn, eps=eps):
                return block(x, c)
        return call

    def __iter__(self):
        for index, scale, replacement in self.schedule:
            block = self.retained[str(index)]
            if replacement and self.mode == "xsa_reuse":
                yield self._xsa(block, self.xsa_eps)
            elif replacement and scale != 1:
                yield self._damped(block, scale)
            else:
                yield block

    def metadata(self):
        return dict(mode=self.mode, removed=self.removed, alpha=self.alpha,
                    damping_policy=("inverse_k" if self.alpha == "inverse_k" else "fixed") if self.mode == "damped" else "none",
                    k_definition="call index of the retained source block, including its original call",
                    replacement_residual_scale=(None if self.alpha == "inverse_k" else self.alpha) if self.mode == "damped" else 1.0,
                    xsa_eps=self.xsa_eps, index_base=0,
                    depth=self.depth, retained=list(map(int, self.retained.keys())),
                    execution_schedule=[dict(slot=slot, source=i, scale=a,
                        replacement=r, iteration=k, xsa=r and self.mode == "xsa_reuse")
                        for slot, (i, a, r), k in zip(self.slots, self.schedule, self.iterations)])


# %% Compact checkpoint loader: same weight format, now restores XSA metadata.
def load_compact_checkpoint(path):
    from models import DiT_XL_2
    payload = torch.load(path, map_location="cpu", weights_only=True, mmap=True)
    meta = payload["schedule"]
    if meta.get("mode") == "custom":
        raise ValueError("Custom checkpoints require the corresponding method builder before loading state_dict")
    if meta.get("index_base", 0) != 0:
        raise ValueError("Expected zero-based block indices")
    with torch.device("meta"):
        model = DiT_XL_2(input_size=32, num_classes=1000)
    model.blocks = ScheduledBlocks(model.blocks, meta["removed"], meta["mode"],
                                   meta["alpha"], xsa_eps=meta.get("xsa_eps", 1e-8))
    if "retained" in meta and meta["retained"] != model.blocks.metadata()["retained"]:
        raise ValueError("Checkpoint original-index mapping is inconsistent")
    model.load_state_dict(payload["state_dict"], strict=True, assign=True)
    return model.eval().requires_grad_(False)



def load_fixed_selection(root):
    selection = json.loads(
        (Path(root) / "profile/selection.json").read_text()
    )
    if selection.get("index_base", 0) != 0:
        raise ValueError("Expected zero-based block indices")
    return selection


def write_fixed_selection(root, removed=(4, 7)):
    removed = list(removed)
    if (
        not removed
        or any(type(i) is not int for i in removed)
        or removed != sorted(set(removed))
        or any(i <= 0 or i >= 28 for i in removed)
    ):
        raise ValueError("Use sorted, unique block indices from 1 through 27")

    path = Path(root) / "profile/selection.json"

    if path.exists():
        selection = load_fixed_selection(root)
        if selection["removed"] != removed:
            raise ValueError(
                "Selection is frozen for this run; create a new run"
            )
        return selection

    selection = {
        "removed": removed,
        "index_base": 0,
        "scoring": "explicitly configured fixed selection",
    }
    write_json(path, selection)
    return selection


def run_method(split, method, name, *, alpha=None, builder=None, settings=None):
    """Use the existing sampler/artifact runner with the frozen selection."""
    if split not in {"validation", "test"}:
        raise ValueError("Use validation or test")
    if Path(name).name != name or name in {".", "..", ""}:
        raise ValueError("name must be a single folder name")
    if (name == "original") != (method == "original" and builder is None):
        raise ValueError("Reserve original for the unmodified baseline")
    if alpha is None:
        alpha = "inverse_k" if method == "damped" else 1.0
    elif alpha != "inverse_k":
        alpha = float(alpha)
    if builder is None and method != "damped" and alpha != 1.0:
        raise ValueError("Only damped mode scales the block residual; use alpha=1.0 for other built-ins")
    root, protocol, config = load_run(RUN_ARGS)
    removed, transform = (), None
    metadata = dict(method=method)
    if builder is not None:
        settings = json.loads(json.dumps(settings or {}))
        source = inspect.getsource(builder)
        metadata.update(settings=settings, source=source,
                        source_sha256=hashlib.sha256(source.encode()).hexdigest())
        transform = lambda model: builder(model, settings)
        mode = "original"
    elif method in {"removal", "loop", "damped", "xsa_reuse"}:
        selection = load_fixed_selection(root)
        removed = selection["removed"]
        metadata["selection"] = selection
        mode = method
    elif method == "original":
        mode = method
    else:
        raise ValueError("Unknown method; pass builder= for a custom method")
    if builder is None:
        metadata["damping_policy"] = (("inverse_k" if alpha == "inverse_k" else "fixed")
                                      if method == "damped" else "none")
        metadata["replacement_residual_scale"] = None if alpha == "inverse_k" else alpha
    return run_variant(config, protocol["splits"][split], root / split / name,
                       mode=mode, removed=removed, alpha=alpha, model_transform=transform,
                       method_metadata=metadata, keep_checkpoint=KEEP_CHECKPOINTS)


# Include the new executed definitions in preparation/resume/evaluation fingerprints.
for _name in ["xsa_filter_heads", "xsa_attention_scope", "load_fixed_selection",
              "write_fixed_selection", "run_method"]:
    if _name not in SHARED_DEFINITION_NAMES:
        SHARED_DEFINITION_NAMES.append(_name)


# %% Focused CPU checks (also included in the notebook's normal test runner).
def test_xsa_projection():
    with torch.random.fork_rng():
        torch.manual_seed(184)
        o, v = torch.randn(2, 3, 5, 8), torch.randn(2, 3, 5, 8)
        z = xsa_filter_heads(o, v)
        torch.testing.assert_close((z * F.normalize(v, dim=-1)).sum(-1),
                                   torch.zeros(2, 3, 5), atol=1e-6, rtol=0)
        # Zero/epsilon-clamped vectors must remain finite; zero V leaves O alone.
        for dtype in [torch.float32, torch.float16, torch.bfloat16]:
            for magnitude in [0.0, 1e-30, 1e-10, 1e-7]:
                small = torch.full_like(v, magnitude).to(dtype)
                result = xsa_filter_heads(o.to(dtype), small)
                assert torch.isfinite(result).all()
                if magnitude == 0:
                    torch.testing.assert_close(result, o.to(dtype), atol=0, rtol=0)


def test_xsa_attention_paths():
    from timm.models.vision_transformer import Attention
    with torch.random.fork_rng():
        torch.manual_seed(185)
        attn = Attention(24, num_heads=3, qkv_bias=True).eval().requires_grad_(False)
        x = torch.randn(2, 5, 24)
        # Both timm attention kernels stay in use; no change when XSA is off.
        paths = [False, True] if hasattr(attn, "fused_attn") else [None]
        for fused in paths:
            if fused is not None:
                attn.fused_attn = fused
            expected_off = attn(x)
            with xsa_attention_scope(attn, enabled=False):
                actual_off = attn(x)
            torch.testing.assert_close(actual_off, expected_off, atol=0, rtol=0)
            with xsa_attention_scope(attn):
                actual_on = attn(x)
            qkv = attn.qkv(x).reshape(2, 5, 3, 3, 8).permute(2, 0, 3, 1, 4)
            q, k, v = qkv.unbind(0)
            q = attn.q_norm(q) if hasattr(attn, "q_norm") else q
            k = attn.k_norm(k) if hasattr(attn, "k_norm") else k
            o = F.scaled_dot_product_attention(q, k, v) if fused else (
                ((q * attn.scale) @ k.transpose(-2, -1)).softmax(-1) @ v)
            expected_on = xsa_filter_heads(o, v).transpose(1, 2).reshape(2, 5, 24)
            norm = getattr(attn, "norm", nn.Identity())
            expected_on = attn.proj_drop(attn.proj(norm(expected_on)))
            torch.testing.assert_close(actual_on, expected_on, atol=1e-6, rtol=1e-5)
            torch.testing.assert_close(attn(x), expected_off, atol=0, rtol=0)
        # Failure during a replacement must not leave XSA active on later calls.
        boundary = getattr(attn, "norm", attn.proj)
        before = (len(attn.qkv._forward_hooks), len(boundary._forward_pre_hooks))
        with pytest.raises(RuntimeError, match="deliberate"):
            with xsa_attention_scope(attn):
                raise RuntimeError("deliberate")
        assert before == (len(attn.qkv._forward_hooks), len(boundary._forward_pre_hooks))
        # Newer timm additionally supports masks, Q/K norms, post-attention norm
        # and an optional attention gate: exercise them without changing forward.
        import inspect
        parameters = inspect.signature(Attention).parameters
        if {"qk_norm", "scale_norm", "gated"} <= parameters.keys():
            normalized = Attention(24, 3, qkv_bias=True, qk_norm=True,
                                   scale_norm=True, gated=True, norm_layer=nn.LayerNorm).eval()
            normalized.requires_grad_(False)
            mask = torch.ones(5, 5, dtype=torch.bool).tril()
            for fused in [False, True]:
                normalized.fused_attn = fused
                off = normalized(x, attn_mask=mask)
                with xsa_attention_scope(normalized, enabled=False):
                    torch.testing.assert_close(normalized(x, attn_mask=mask), off, atol=0, rtol=0)
                with xsa_attention_scope(normalized):
                    actual = normalized(x, attn_mask=mask)
                q, k, v = normalized.qkv(x).reshape(2, 5, 3, 3, 8).permute(2, 0, 3, 1, 4).unbind(0)
                q, k = normalized.q_norm(q), normalized.k_norm(k)
                if fused:
                    o = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
                else:
                    logits = ((q * normalized.scale) @ k.transpose(-2, -1)).masked_fill(~mask, -float("inf"))
                    o = logits.softmax(-1) @ v
                heads = xsa_filter_heads(o, v).transpose(1, 2).reshape(2, 5, 24)
                expected = normalized.norm(heads) * normalized.gate(x).sigmoid()
                expected = normalized.proj_drop(normalized.proj(expected))
                torch.testing.assert_close(actual, expected, atol=1e-6, rtol=1e-5)


def test_xsa_schedule_and_registration(monkeypatch):
    from timm.models.vision_transformer import Attention

    class ProbeBlock(nn.Module):
        def __init__(self, index):
            super().__init__()
            self.index = index
            self.attn = Attention(12, num_heads=3, qkv_bias=True)
            self.inputs = []

        def forward(self, x, c):
            self.inputs.append((x.detach().clone(), c))
            return x + self.attn(x) + c[:, None, :]

    with torch.random.fork_rng():
        torch.manual_seed(186)
        blocks = [ProbeBlock(i) for i in range(28)]
        schedule = ScheduledBlocks(blocks, [4, 7], "xsa_reuse", alpha=1.0)
        schedule.eval().requires_grad_(False)
        expected_sources = [0, 1, 2, 3, 3, 5, 6, 6, *range(8, 28)]
        metadata = schedule.metadata()
        assert [s["source"] for s in metadata["execution_schedule"]] == expected_sources
        assert [s["slot"] for s in metadata["execution_schedule"] if s["xsa"]] == [4, 7]
        assert metadata["index_base"] == 0 and metadata["retained"] == [i for i in range(28) if i not in [4, 7]]
        assert schedule.retained["3"] is blocks[3] and schedule.retained["6"] is blocks[6]
        retained_ids = {id(p) for b in blocks if b.index not in [4, 7] for p in b.parameters()}
        discarded_ids = {id(p) for b in [blocks[4], blocks[7]] for p in b.parameters()}
        registered_ids = {id(p) for p in schedule.parameters()}
        assert registered_ids == retained_ids and not registered_ids & discarded_ids
        assert len(schedule.retained) == 26 and len(schedule) == 28
        assert all(not p.requires_grad for p in schedule.parameters())
        assert not any(key.split(".")[1] in {"4", "7"} for key in schedule.state_dict())

        events, qkv_inputs, ordinary_outputs = [], {3: [], 6: []}, {}
        original_filter = xsa_filter_heads

        def record_filter(o, v, eps=1e-8):
            events.append(active_slot[0])
            return original_filter(o, v, eps)

        monkeypatch.setitem(globals(), "xsa_filter_heads", record_filter)
        handles = []
        for index in [3, 6]:
            handles.append(blocks[index].attn.qkv.register_forward_pre_hook(
                lambda module, args, index=index: qkv_inputs[index].append(args[0].detach().clone())))
        x, c = torch.randn(1, 5, 12) * 0.01, torch.randn(1, 12) * 0.01
        active_slot = [None]
        try:
            for slot, call in enumerate(schedule):
                active_slot[0] = slot
                prior = x.clone()
                x = call(x, c)
                if slot in [3, 6]:
                    ordinary_outputs[slot] = x.clone()
                # Each additional call receives the immediately preceding output.
                if slot in [4, 7]:
                    source = 3 if slot == 4 else 6
                    torch.testing.assert_close(blocks[source].inputs[-1][0], prior)
                    assert blocks[source].inputs[-1][1] is c
            assert events == [4, 7]
            assert torch.isfinite(x).all()
            for index in [3, 6]:
                assert len(qkv_inputs[index]) == 2
                assert not torch.equal(*qkv_inputs[index])
                # The ordinary retained call remains the exact original block path.
                input_x, input_c = blocks[index].inputs[0]
                ordinary = blocks[index](input_x, input_c)
                torch.testing.assert_close(ordinary, ordinary_outputs[index], atol=0, rtol=0)
        finally:
            for handle in handles:
                handle.remove()


def test_inverse_k_damping():
    class AddBlock(nn.Module):
        def forward(self, x, c):
            return x + 1

    def evaluate(schedule):
        x = torch.zeros(1)
        for call in schedule:
            x = call(x, None)
        return x

    schedule = ScheduledBlocks([AddBlock() for _ in range(6)], [1, 2, 4], "damped")
    entries = schedule.metadata()["execution_schedule"]
    assert [entry["source"] for entry in entries] == [0, 0, 0, 3, 3, 5]
    assert [entry["iteration"] for entry in entries] == [1, 2, 3, 1, 2, 1]
    assert [entry["scale"] for entry in entries] == [1.0, 0.5, 1 / 3, 1.0, 0.5, 1.0]
    assert schedule.metadata()["damping_policy"] == "inverse_k"
    assert schedule.metadata()["replacement_residual_scale"] is None
    torch.testing.assert_close(evaluate(schedule), torch.tensor([13 / 3]))
    # Calling a forward again starts from the same precomputed k values.
    torch.testing.assert_close(evaluate(schedule), evaluate(schedule), atol=0, rtol=0)
    fixed = ScheduledBlocks([AddBlock() for _ in range(6)], [1, 2, 4], "damped", alpha=0.25)
    assert fixed.metadata()["damping_policy"] == "fixed"
    torch.testing.assert_close(evaluate(fixed), torch.tensor([3.75]))
    ordinary = ScheduledBlocks([AddBlock() for _ in range(6)], [1, 2, 4], "loop")
    torch.testing.assert_close(evaluate(ordinary), torch.tensor([6.0]))


def test_xsa_checkpoint_and_fixed_selection(tmp_path, monkeypatch):
    import sys
    import types
    from timm.models.vision_transformer import Attention

    class SmallModel(nn.Module):
        def __init__(self, **kwargs):
            super().__init__()
            self.blocks = nn.ModuleList([nn.ModuleDict({"attn": Attention(12, 3)}) for _ in range(28)])

    monkeypatch.setitem(sys.modules, "models", types.SimpleNamespace(DiT_XL_2=SmallModel))
    model = SmallModel()
    model.blocks = ScheduledBlocks(model.blocks, [4, 7], "xsa_reuse", 1.0, xsa_eps=1e-7)
    path = tmp_path / "compact.pt"
    stats = checkpoint_stats(model, path, keep=True)
    loaded = load_compact_checkpoint(path)
    assert loaded.blocks.metadata() == model.blocks.metadata()
    assert stats["unique_blocks"] == 26 and stats["block_evaluations_per_forward"] == 28
    assert all(not p.requires_grad for p in loaded.parameters())
    for key, value in model.state_dict().items():
        torch.testing.assert_close(loaded.state_dict()[key], value, atol=0, rtol=0)
    selection = write_fixed_selection(tmp_path)
    assert selection["removed"] == [4, 7] and selection["index_base"] == 0
    assert load_fixed_selection(tmp_path) == selection
    # Verify the public entry point uses shared config/pairs and fixed indices.
    config, pairs = {"device": "cpu"}, [[207, 10016]]
    protocol = {"splits": {"validation": pairs}}
    monkeypatch.setitem(globals(), "RUN_ARGS", object())
    monkeypatch.setitem(globals(), "KEEP_CHECKPOINTS", False)
    monkeypatch.setitem(globals(), "load_run", lambda args: (tmp_path, protocol, config))
    calls = []

    def capture_run(config_arg, pairs_arg, output, **kwargs):
        calls.append((config_arg, pairs_arg, output, kwargs))

    monkeypatch.setitem(globals(), "run_variant", capture_run)
    for mode in ["removal", "loop", "xsa_reuse"]:
        run_method("validation", mode, mode, alpha=1.0)
        actual_config, actual_pairs, output, kwargs = calls[-1]
        assert actual_config is config and actual_pairs is pairs
        assert output == tmp_path / "validation" / mode
        assert kwargs["mode"] == mode and kwargs["removed"] == [4, 7]
        assert kwargs["alpha"] == 1.0 and kwargs["model_transform"] is None
    with pytest.raises(ValueError, match="alpha=1.0"):
        run_method("validation", "xsa_reuse", "bad_alpha", alpha="inverse_k")
    selection["removed"] = [3, 6]
    write_json(tmp_path / "profile/selection.json", selection)
    with pytest.raises(ValueError, match="frozen"):
        write_fixed_selection(tmp_path)


# %% Combined notebook test runner. No checkpoint or image downloads.
from tempfile import TemporaryDirectory


def run_notebook_tests():
    tests = [
        ("protocol splits/hash", lambda path, patch: test_protocol_has_balanced_disjoint_splits_and_stable_hash(path)),
        ("preprocessing/noise", lambda path, patch: test_preprocessing_and_initial_noise_are_fixed()),
        ("runner/evaluator artifacts", lambda path, patch: test_shared_runner_and_evaluator_reject_incomplete_or_mismatched_runs(path, patch)),
        ("custom builder", lambda path, patch: test_custom_builder_keeps_shared_preparation_and_freezes_model(patch)),
        ("inverse-k damping/reset/fixed compatibility", lambda path, patch: test_inverse_k_damping()),
        ("XSA projection/zero values", lambda path, patch: test_xsa_projection()),
        ("XSA attention on/off and hook cleanup", lambda path, patch: test_xsa_attention_paths()),
        ("XSA schedule/shared frozen weights", lambda path, patch: test_xsa_schedule_and_registration(patch)),
        ("XSA checkpoint/fixed selection", lambda path, patch: test_xsa_checkpoint_and_fixed_selection(path, patch)),
    ]
    for pairs in [[], [(1, 3), (2, 3)], [(1000, 3)], [(1.5, 3)], [(True, 3)]]:
        tests.append((f"invalid input {pairs}", lambda path, patch, pairs=pairs:
                      test_input_validation_rejects_ambiguous_samples(pairs)))
    for name, test in tests:
        with TemporaryDirectory() as directory, pytest.MonkeyPatch.context() as patch:
            test(Path(directory), patch)
        print("PASS:", name)
    print(f"All {len(tests)} notebook test cases passed.")


# Definitions only. Continue with tests, configuration and block selection below.


## 3. RUN TESTS — verify the live notebook code

Run all definition cells first, including the XSA extension immediately above. This runs 14 CPU cases covering protocol, preprocessing, runner artifacts, custom builders, XSA attention, weight sharing and checkpoint restoration. No pretrained downloads are needed. Stop if any assertion fails.


In [32]:
# RUN TESTS — CPU only, no generated .py files.
from tempfile import TemporaryDirectory

run_notebook_tests()


PASS: protocol splits/hash
PASS: preprocessing/noise
original: 1/2, 0.00 s
original: 2/2, 0.00 s
original: 1/2, 0.00 s
original: 2/2, 0.00 s
PASS: runner/evaluator artifacts
PASS: custom builder
PASS: inverse-k damping/reset/fixed compatibility
PASS: XSA projection/zero values
PASS: XSA attention on/off and hook cleanup
PASS: XSA schedule/shared frozen weights
PASS: XSA checkpoint/fixed selection
PASS: invalid input []
PASS: invalid input [(1, 3), (2, 3)]
PASS: invalid input [(1000, 3)]
PASS: invalid input [(1.5, 3)]
PASS: invalid input [(True, 3)]
All 14 notebook test cases passed.


## 4. CONFIGURE — device, source and experiment settings

Run after tests pass. `DEVICE="auto"` selects CUDA, then Apple MPS, then CPU; you can explicitly choose `"cuda:0"`, `"mps"` or `"cpu"`. Full DiT-XL inference on CPU is slow. The DiT checkpoint download is about 2.5 GiB, and the VAE is downloaded on first generation.

Choose a unique member/run name. The XSA comparison defaults to fixed zero-based block indices `[4, 7]` for every compressed method. Set `CALIBRATE_BLOCKS=True` to select blocks by calibration instead. Re-running this cell creates a new run path unless `RESUME_RUN` is set. **Create a new run for the Imagenette class protocol; existing eight-class outputs cannot be mixed with it.** After changing shared code, use a new run for generation. To plot existing results without inference or environment checks, run section 12 directly. To evaluate compatible existing outputs, set `RESUME_RUN` here, then run section 8; generation cells must not be rerun for completed variants.


In [33]:
# CONFIGURE — does not generate images.
import json
from datetime import datetime, timezone
import torch
from IPython.display import display, Image

DEVICE = "auto"
RESUME_RUN = None  # Leave None for the new Imagenette protocol; old eight-class runs differ.

protocol = load_protocol(PROTOCOL)
REPO = WORKSPACE / "third_party/DiT"
if not (REPO / "models.py").is_file():
    REPO.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/facebookresearch/DiT.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", protocol["settings"]["dit_commit"]], check=True)
MEMBER = "pairing_pruned"  # Use your name for independent runs.
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_ROOT = (Path(RESUME_RUN).expanduser().resolve() if RESUME_RUN
            else OUTPUT_BASE / f"{MEMBER}_{RUN_ID}")
CHECKPOINT = WORKSPACE / "checkpoints/DiT-XL-2-256x256.pt"
ALPHA = "inverse_k"  # Reused calls use 1/2, 1/3, ...; original call counts as k=1.
KEEP_CHECKPOINTS = False  # True retains several GB per variant.

if RESUME_RUN:
    saved_config = json.loads((RUN_ROOT / "config.json").read_text())
    DEVICE = saved_config["device"]
    CHECKPOINT = Path(saved_config["checkpoint"])
device = select_device(DEVICE)
RUN_ARGS = SimpleNamespace(output=RUN_ROOT, repo=REPO, protocol=PROTOCOL,
                           device=str(device), checkpoint=CHECKPOINT)

print("Device:", device, "| PyTorch:", torch.__version__)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(device))
print("Run directory:", RUN_ROOT)
print("ImageNet class ids:", list(CLASS_LABELS))
print("Images per split:", {key: len(value) for key, value in protocol["splits"].items()})

PAIRING_MODE = "pruned"  # This notebook runs only this mode.
SHARED_PAIRING_DESIGN = None  # Optional path to teammate's pairing_experiments/design.json.


Device: mps | PyTorch: 2.15.0
Run directory: /Users/matthewlee/Desktop/CORNELL/CS5787/cs5787-deep-learning/team_runs/pairing_pruned_20261124T075035681906Z
ImageNet class ids: [0, 217, 482, 491, 497, 566, 569, 571, 574, 701]
Images per split: {'calibration': 20, 'validation': 20, 'test': 5000}


HEAD is now at ed81ce2 Update README.md


## 5. RUN PREPARE — freeze shared inputs

Downloads the pretrained DiT checkpoint if needed and verifies its SHA-256. Records protocol, shared-code hash, environment and package versions. Requires a **new** output directory; skip this cell when resuming an already prepared run.

In [34]:
# RUN PREPARE — skip downloading/preparing again when resuming.
if RESUME_RUN:
    load_run(RUN_ARGS)  # Verify the saved code, environment and inputs.
    print("Resumed:", RUN_ROOT)
else:
    prepare(RUN_ARGS)


Prepared shared inputs: /Users/matthewlee/Desktop/CORNELL/CS5787/cs5787-deep-learning/team_runs/pairing_pruned_20261124T075035681906Z


## 6. METRICS — shared LPIPS/FID helpers and reference settings

Definitions only. Evaluation runs after generation and diagnostics in section 7D.


In [35]:
# 8B - FIDs & TPIPS
import gc
import hashlib
import importlib.metadata
import json
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import torch


def _extra_version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return "not-installed"


def _extra_read_image(path, size):
    image = torch.load(path, map_location="cpu", weights_only=True)
    if (not isinstance(image, torch.Tensor) or image.shape != (1, 3, size, size)
            or not image.is_floating_point() or not torch.isfinite(image).all()
            or image.min() < -1.00001 or image.max() > 1.00001):
        raise ValueError(f"Invalid saved RGB image tensor: {path}")
    return image.detach().float().clamp(-1, 1)


def _extra_read_artifacts(root, names):
    """Validate paired inputs using the existing manifests, samples and LPIPS CSVs."""
    names = list(names)
    if not names or len(set(names)) != len(names):
        raise ValueError("Use nonempty, unique variant names")
    if any(not isinstance(n, str) or Path(n).name != n or n in {".", "..", ""} for n in names):
        raise ValueError("Each variant name must be a single folder name")
    summary = pd.read_csv(root / "results.csv")
    paired = pd.read_csv(root / "paired_metrics.csv")
    if summary["variant"].duplicated().any():
        raise ValueError("Duplicate variants in results.csv")
    baseline = json.loads((root / "original/manifest.json").read_text())
    pairs = baseline["pairs"]
    if not pairs:
        raise ValueError("Empty baseline sample list")
    expected = pd.DataFrame([(i, *pair) for i, pair in enumerate(pairs)],
                            columns=["index", "label", "seed"])
    samples_by_name = {}
    for name in dict.fromkeys(["original", *names]):
        manifest = json.loads((root / name / "manifest.json").read_text())
        for key in ["config", "pairs", "shared_code_sha256", "runtime"]:
            if key not in manifest or manifest[key] != baseline.get(key):
                raise ValueError(f"Mismatched generation {key}: {name}")
        samples = pd.read_csv(root / name / "samples.csv")
        if not samples[["index", "label", "seed"]].equals(expected):
            raise ValueError(f"Incomplete or mismatched sample order: {name}")
        samples_by_name[name] = samples
        if name in names:
            rows = paired.loc[paired["variant"] == name]
            if not rows[["index", "label", "seed"]].reset_index(drop=True).equals(expected):
                raise ValueError(f"Incomplete paired_metrics.csv rows: {name}")
            result = summary.loc[summary["variant"] == name]
            if len(result) != 1 or int(result.iloc[0]["images"]) != len(expected):
                raise ValueError(f"Missing or inconsistent results.csv entry: {name}")
    summary = summary.set_index("variant").loc[names].reset_index()
    paired = pd.concat([paired.loc[paired["variant"] == name] for name in names], ignore_index=True)
    return summary, paired, baseline, samples_by_name


# Imagenette reference preparation uses torchvision, which is already installed.
# Download: full-resolution Imagenette v2 (~1.45 GiB archive), not all ImageNet.
def prepare_imagenette_reference(source_root, prepared_root, pairs, *,
                                 images_per_class=500, image_size=256, seed=0):
    """Export a deterministic, class-balanced real-image reference and reuse it.

    Uses the dataset's TRAIN split as real FID reference data; DiT is not trained.
    Generated labels come from the saved run manifest, never assumed local ids.
    Source files/archives live outside the prepared image folder so Clean-FID
    cannot accidentally include raw images or grids in the reference.
    """
    from collections import Counter
    import random
    from PIL import Image as PILImage, ImageOps

    pairs = validate_pairs(pairs)
    counts = Counter(label for label, seed_value in pairs)
    class_map = {label: (synset, name) for label, synset, name in IMAGENETTE_CLASSES}
    labels = sorted(counts)
    if set(labels) - class_map.keys():
        raise ValueError("Run contains classes outside Imagenette; prepare a new run using CLASS_LABELS")
    if len(set(counts.values())) != 1:
        raise ValueError("Use balanced generated class counts to match the real reference")
    if type(images_per_class) is not int or images_per_class < 2:
        raise ValueError("images_per_class must be an integer >= 2")
    if type(image_size) is not int or image_size <= 0:
        raise ValueError("image_size must be a positive integer")
    if type(seed) is not int or seed < 0:
        raise ValueError("Reference seed must be a nonnegative integer")
    settings = dict(
        dataset="Imagenette v2", source_split="train", source_size="full",
        source_url="https://s3.amazonaws.com/fast-ai-imageclas/imagenette2.tgz",
        archive_md5="fe2fc210e6bb7c5664d602c3cd71e612",
        labels=labels, images_per_class=images_per_class, image_size=image_size,
        seed=seed, preprocessing="RGB; aspect-preserving bicubic resize and center crop; PNG",
    )
    digest = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()[:16]
    output = Path(prepared_root) / f"imagenette2_train_{images_per_class}perclass_{digest}"
    manifest_path = output / "reference_manifest.json"
    expected = {f"class_{label:04d}_{i:04d}.png"
                for label in labels for i in range(images_per_class)}
    description = (
        f"Imagenette v2 full-resolution train split; ImageNet labels={labels}; "
        f"{images_per_class} real images per class, selected with seed={seed}; "
        f"RGB; aspect-preserving bicubic resize and center crop to {image_size}x{image_size}; PNG."
    )
    if manifest_path.exists():
        saved = json.loads(manifest_path.read_text())
        actual = {p.name for p in output.glob("*.png") if p.is_file()}
        if (saved.get("settings") != settings or actual != expected
                or any((output / filename).stat().st_size == 0 for filename in expected)):
            raise ValueError(f"Cached Imagenette reference is incomplete or inconsistent: {output}")
        print("Reusing real FID reference:", output, flush=True)
        return output, description

    from torchvision.datasets import Imagenette
    source_root = Path(source_root)
    print("Preparing Imagenette v2 full-resolution reference (one-time download).", flush=True)
    Imagenette(root=str(source_root), split="train", size="full", download=True)
    selections = {}
    extensions = {".jpeg", ".jpg", ".png"}
    for label in labels:
        synset, name = class_map[label]
        folder = source_root / "imagenette2" / "train" / synset
        files = sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in extensions)
        if len(files) < images_per_class:
            raise ValueError(f"{name} ({label}) has {len(files)} real images; requested {images_per_class}")
        selections[label] = sorted(random.Random(seed + label).sample(files, images_per_class))
    output.mkdir(parents=True, exist_ok=True)
    if {p.name for p in output.glob("*.png")} - expected:
        raise ValueError(f"Unexpected images in reference folder: {output}")
    inventory = []
    for label in labels:
        for i, source in enumerate(selections[label]):
            filename = f"class_{label:04d}_{i:04d}.png"
            with PILImage.open(source) as image:
                image = ImageOps.fit(image.convert("RGB"), (image_size, image_size),
                                     method=PILImage.Resampling.BICUBIC, centering=(0.5, 0.5))
                image.save(output / filename)
            inventory.append(dict(file=filename, label=label,
                                  source=str(source.relative_to(source_root))))
        print(f"Real reference class {label}: {images_per_class} images", flush=True)
    write_json(manifest_path, dict(settings=settings, images=len(inventory), inventory=inventory))
    print("Saved real FID reference:", output, flush=True)
    return output, description


@torch.inference_mode()
def evaluate_fid_tpips(split_root, names, *, fid_reference_dir=None,
                       fid_reference_description=None,
                       tpips_factors=("overall", "shape", "texture"),
                       fid_device="cpu", tpips_device="cuda:0",
                       fid_batch_size=16, tpips_model_path=None):
    """Append optional metrics to separate extended CSVs; lower is better for both.

    FID uses pooled Clean-FID Inception features against a REAL-image directory.
    Only manifest-listed image_NNNN.png files are evaluated: grids are excluded.
    TPIPS uses official embedding-model distance against matched original images,
    from saved FP32 tensors with the explicit numeric range [-1, 1]. Prompts are
    separate metrics. With both extras disabled, export the existing LPIPS results.
    Install optional metric dependencies only after the base LPIPS evaluation.
    """
    root = Path(split_root).expanduser().resolve()
    names = list(names)
    factors = list(tpips_factors)
    if any(not isinstance(f, str) or not f.strip() for f in factors) or len(set(factors)) != len(factors):
        raise ValueError("Use unique, nonempty TPIPS factor strings")
    columns = ["tpips_" + re.sub(r"[^a-z0-9]+", "_", factor.lower()).strip("_") for factor in factors]
    if len(set(columns)) != len(columns) or "tpips_" in columns:
        raise ValueError("TPIPS factor names produce empty or colliding CSV column names")
    summary, paired, baseline, samples_by_name = _extra_read_artifacts(root, names)
    if fid_reference_dir is None and not factors:
        print("FID and TPIPS disabled; exporting LPIPS results only.", flush=True)
    indices = samples_by_name["original"]["index"].tolist()
    size = baseline["config"]["image_size"]
    metadata = dict(
        generation_code_sha256=baseline["shared_code_sha256"],
        generation_runtime=baseline["runtime"], variants=names,
        images_per_variant=len(indices), lower_is_better=True,
        packages={p: _extra_version(p) for p in ["torch", "torchvision", "numpy",
                   "scipy", "clean-fid", "tpips", "transformers", "pillow"]},
        fid=None, tpips=None,
    )

    if fid_reference_dir is not None:
        from cleanfid import fid
        from cleanfid.features import build_feature_extractor
        reference = Path(fid_reference_dir).expanduser().resolve()
        if not reference.is_dir():
            raise FileNotFoundError(f"Real-image FID reference folder missing: {reference}")
        if not fid_reference_description:
            raise ValueError("Describe the FID reference dataset, class mixture, split and preprocessing")
        if reference == root or root in reference.parents or reference in root.parents:
            raise ValueError("FID reference must be real images outside the generated split directory")
        extensions = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}
        reference_files = sorted(p for p in reference.rglob("*") if p.is_file() and p.suffix.lower() in extensions)
        if len(indices) < 2 or len(reference_files) < 2:
            raise ValueError("FID needs at least two generated and two reference images")
        if len(indices) < 2049 or len(reference_files) < 2049:
            warnings.warn("Small-sample FID: 2048-D covariance is rank-deficient below 2049 images. "
                          "Treat this as a noisy pilot diagnostic, not a benchmark-quality FID.", stacklevel=2)
        generated_files = {
            name: [root / name / f"image_{int(i):04d}.png" for i in indices]
            for name in names
        }
        for files in generated_files.values():
            for path in files:
                if not path.is_file():
                    raise FileNotFoundError(path)
        model = build_feature_extractor("clean", device=torch.device(fid_device), use_dataparallel=False)
        try:
            def features(files):
                values = fid.get_files_features(
                    [str(p) for p in files], model=model, num_workers=0,
                    batch_size=fid_batch_size, device=torch.device(fid_device), mode="clean")
                if values.ndim != 2 or values.shape[0] != len(files) or not np.isfinite(values).all():
                    raise ValueError("Invalid Clean-FID features")
                return values

            reference_features = features(reference_files)
            fid_scores = {}
            for name in names:
                score = float(fid.fid_from_feats(features(generated_files[name]), reference_features))
                if not np.isfinite(score):
                    raise ValueError(f"Nonfinite FID: {name}")
                fid_scores[name] = score
                print(f"FID {name}: {score:.6f}", flush=True)
            summary["fid"] = summary["variant"].map(fid_scores)
        finally:
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        inventory = [dict(path=str(p.relative_to(reference)), bytes=p.stat().st_size) for p in reference_files]
        (root / "fid_reference_inventory.json").write_text(json.dumps(inventory, indent=2) + "\n")
        metadata["fid"] = dict(
            implementation="clean-fid", mode="clean", features="inception_v3_2048",
            reference_dir=str(reference), reference_description=fid_reference_description,
            reference_images=len(reference_files), generated_images=len(indices),
            reference_inventory_sha256=hashlib.sha256(json.dumps(inventory, sort_keys=True).encode()).hexdigest(),
            inventory_hash_scope="relative filenames and file sizes, not image content",
            input="only manifest-listed image_NNNN.png; grids excluded", device=fid_device,
            small_sample_pilot=len(indices) < 2049 or len(reference_files) < 2049,
        )

    if factors:
        import tpips
        metric = tpips.load_model("embedding", model_path=tpips_model_path, device=tpips_device)
        metric.eval().requires_grad_(False)
        try:
            for name in names:
                values = {column: [] for column in columns}
                for i in indices:
                    ref = _extra_read_image(root / "original" / f"image_{int(i):04d}.pt", size)
                    image = _extra_read_image(root / name / f"image_{int(i):04d}.pt", size)
                    for factor, column in zip(factors, columns):
                        score = float(metric.distance(ref, image, factor=factor, value_range=(-1.0, 1.0)).item())
                        if not np.isfinite(score):
                            raise ValueError(f"Nonfinite TPIPS: {name}, sample={i}, factor={factor}")
                        values[column].append(score)
                for column, scores in values.items():
                    paired.loc[paired["variant"] == name, column] = scores
                    summary.loc[summary["variant"] == name, column + "_mean"] = float(np.mean(scores))
                    summary.loc[summary["variant"] == name, column + "_std"] = float(np.std(scores, ddof=1)) if len(scores) > 1 else 0.0
                print(f"TPIPS finished: {name}", flush=True)
        finally:
            del metric
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        metadata["tpips"] = dict(
            model_type="embedding", model_path=str(tpips_model_path) if tpips_model_path else "sywang/TPIPS-Embed-Qwen3VL-8B",
            factors=dict(zip(columns, factors)), paired_reference="original",
            input="saved FP32 RGB tensors; TPIPS value_range=(-1,1); official model preprocessing",
            metric="official model.distance (1 - cosine similarity)", device=tpips_device,
            interpretation="aspect-conditioned deviation from original, not absolute generation quality",
        )

    # Preserve the original LPIPS evaluation files and generation measurements.
    summary.to_csv(root / "results_extended.csv", index=False)
    paired.to_csv(root / "paired_metrics_extended.csv", index=False)
    (root / "evaluation_extended.json").write_text(json.dumps(metadata, indent=2, allow_nan=False) + "\n")
    print("Saved:", root / "results_extended.csv")
    return summary


In [36]:
# METRIC SETTINGS — Imagenette FID enabled, TPIPS disabled.
USE_IMAGENETTE_REFERENCE = True
IMAGENETTE_SOURCE_ROOT = WORKSPACE / "fid_references" / "source"
IMAGENETTE_PREPARED_ROOT = WORKSPACE / "fid_references" / "prepared"
FID_REFERENCE_DIR = None  # Filled automatically from the evaluated run's classes.
FID_REFERENCE_DESCRIPTION = None
TPIPS_FACTORS = ()  # No TPIPS package/weights are loaded with this default.
FID_DEVICE = "cpu"
TPIPS_DEVICE = str(select_device("auto"))


## 7. All five pruned variants

| Pairing | Removed blocks | Replay sources in the damped notebook |
|---|---|---|
| Feature / before | `[anchor-1]` | `[anchor]` |
| Feature / after | `[anchor+1]` | `[anchor]` |
| Position / late | `[22,23]` | `[20,21]` |
| Position / middle | `[16,17]` | `[14,15]` |
| Position / early | `[11,12]` | `[9,10]` |

The anchor is the largest mean feature change among interior blocks, reversing the original selection ranking using calibration samples only. The five removal sets match across notebooks. **Pruned mode executes no replays.** Damped mode replays each source once, immediately after the loop sequence's first pass, using `x <- x + 0.5*(block(x,c)-x)`.

Feature trials remove one block; position trials remove two. Damped runs execute 28 blocks per forward; their prune-only controls execute 27 or 26. Compare removal budgets within each group. Outputs live in `RUN_ROOT/pairing_experiments/`. Complete matching runs are skipped; partial/mismatched runs raise an error.


### 7A. DEFINE — schedule, calibration, isolated runner and paired metrics


In [37]:
# PAIRING DEFINITIONS — independent of the old selection and member variants.
import inspect
import hashlib
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn


def pairing_schedule(depth, removed, looped):
    """Natural retained order, then one ordered replay immediately after looped[-1]."""
    removed, looped = list(removed), list(looped)
    for values in (removed, looped):
        if values != sorted(set(values)) or any(type(i) is not int or not 0 <= i < depth for i in values):
            raise ValueError('Indices must be unique, ascending, zero-based integers in range')
    if not removed or removed != list(range(removed[0], removed[-1] + 1)):
        raise ValueError('Remove exactly one contiguous span')
    if set(removed) & set(looped) or (looped and len(looped) != len(removed)):
        raise ValueError('Replay retained blocks only; extra executions must equal removals')
    schedule = []
    for index in range(depth):
        if index in removed:
            continue
        schedule.append((index, 1., False))
        if looped and index == looped[-1]:
            schedule.extend((source, .5, True) for source in looped)
    return schedule


def pairing_specs(anchor):
    if type(anchor) is not int or not 1 <= anchor <= 26:
        raise ValueError('Anchor must be an interior DiT-XL block (1..26)')
    return [
        dict(name='feature_before', group='feature', removed=[anchor-1], looped=[anchor]),
        dict(name='feature_after', group='feature', removed=[anchor+1], looped=[anchor]),
        dict(name='position_late', group='position', removed=[22,23], looped=[20,21]),
        dict(name='position_middle', group='position', removed=[16,17], looped=[14,15]),
        dict(name='position_early', group='position', removed=[11,12], looped=[9,10]),
    ]


class PairingBlocks(nn.Module):
    def __init__(self, blocks, removed, looped):
        super().__init__()
        blocks = list(blocks)
        self.depth = len(blocks)
        self.removed, self.looped = list(removed), list(looped)
        self.schedule = pairing_schedule(self.depth, removed, looped)
        self.retained = nn.ModuleDict({str(i): block for i, block in enumerate(blocks) if i not in removed})

    def __len__(self):
        return len(self.schedule)

    def __iter__(self):
        for source, scale, repeat in self.schedule:
            block = self.retained[str(source)]
            yield ScheduledBlocks._damped(block, scale) if repeat else block

    def metadata(self):
        return dict(mode='custom', builder='build_pairing_model', depth=self.depth,
                    removed=self.removed, looped=self.looped,
                    retained=list(map(int, self.retained.keys())),
                    damping_policy='inverse_k', alpha='inverse_k',
                    order='retained ascending; replay loop sequence after its first pass',
                    execution_schedule=[dict(source=i, scale=a, replacement=r, iteration=2 if r else 1)
                                        for i, a, r in self.schedule])


def build_pairing_model(model, settings):
    model.blocks = PairingBlocks(model.blocks.retained.values(), settings['removed'], settings['looped'])
    return model


def pairing_source_hash():
    objects = [pairing_schedule, pairing_specs, build_pairing_model,
               select_pairing_anchor, pairing_portable_config, prepare_pairing_experiments, pairing_complete,
               run_pairing_experiments]
    objects += [v for v in vars(PairingBlocks).values() if inspect.isfunction(v)]
    return hashlib.sha256('\n'.join(inspect.getsource(o) for o in objects).encode()).hexdigest()


def select_pairing_anchor(frame):
    # Same per-timestep mean as summarize_profile, reversed ranking (not 1/score).
    matrix = frame.groupby(['block', 'timestep'])['mean_cfg'].mean().unstack()
    if list(matrix.index) != list(range(28)) or matrix.isna().any().any() or not np.isfinite(matrix.to_numpy()).all():
        raise ValueError('Require complete finite calibration for all 28 blocks')
    scores = matrix.mean(axis=1).rename('mean_feature_change')
    # Match SELECT BLOCKS' protected endpoints for the anchor. Neighbor removal
    # may include 0 or 27; the custom schedule supports either endpoint.
    anchor = int(scores.loc[1:26].sort_values(ascending=False, kind='stable').index[0])
    return anchor, scores


def prepare_pairing_experiments():
    if PAIRING_MODE not in {'damped', 'pruned'}:
        raise ValueError('PAIRING_MODE must be damped or pruned')
    root, protocol, config = load_run(RUN_ARGS)
    output = root / 'pairing_experiments'
    output.mkdir(exist_ok=True)
    frozen_path = output / 'design.json'
    contract = dict(config=config, mode=PAIRING_MODE, calibration_pairs=protocol['splits']['calibration'],
                    source_sha256=pairing_source_hash())
    if frozen_path.exists():
        design = json.loads(frozen_path.read_text())
        if design['contract'] != contract:
            raise ValueError('Pairing code/settings changed; use a new prepared run')
        return output, protocol, config, design
    if SHARED_PAIRING_DESIGN is not None:
        shared = json.loads(Path(SHARED_PAIRING_DESIGN).expanduser().read_text())
        if (pairing_portable_config(shared['contract']['config']) != pairing_portable_config(config)
                or shared['contract']['calibration_pairs'] != protocol['splits']['calibration']
                or shared['contract']['source_sha256'] != pairing_source_hash()
                or shared['specs'] != pairing_specs(shared['anchor'])):
            raise ValueError('Shared design does not match this protocol/code')
        design = {**shared, 'contract': contract}
        write_json(frozen_path, design)
        return output, protocol, config, design
    profile_dir = output / 'profile'
    profile_path = profile_dir / 'feature_changes.csv'
    if profile_path.exists():
        calibration = json.loads((profile_dir / 'calibration_manifest.json').read_text())
        if calibration != dict(config=config, pairs=protocol['splits']['calibration']):
            raise ValueError('Calibration settings differ')
        frame = pd.read_csv(profile_path)
    else:
        frame = calibrate(config, protocol['splits']['calibration'], profile_dir)
    anchor, scores = select_pairing_anchor(frame)
    scores.to_csv(profile_dir / 'block_scores.csv', index_label='block')
    design = dict(contract=contract, anchor=anchor, specs=pairing_specs(anchor),
                  budgets='feature: 1 removed/1 replay; position: 2 removed/2 replays',
                  update='x <- x + (block(x,c)-x)/2 on each second execution')
    write_json(frozen_path, design)
    return output, protocol, config, design


def pairing_complete(folder, config, pairs, metadata):
    """Skip only complete runs with the same settings; never mix partial outputs."""
    if not folder.exists():
        return False
    manifest_path = folder / 'manifest.json'
    if not manifest_path.exists():
        raise ValueError(f'Incomplete run: {folder}; move it aside before retrying')
    manifest = json.loads(manifest_path.read_text())
    if (manifest['config'] != config or manifest['pairs'] != pairs
            or manifest['method_metadata'] != metadata
            or manifest['shared_code_sha256'] != shared_code_hash()
            or manifest['runtime'] != environment(config['device'])):
        raise ValueError(f'Existing run differs: {folder}')
    expected = pd.DataFrame([(i,*p) for i,p in enumerate(pairs)], columns=['index','label','seed'])
    if not (folder / 'samples.csv').exists():
        raise ValueError(f'Incomplete run: {folder}')
    actual = pd.read_csv(folder / 'samples.csv')
    if not actual[['index','label','seed']].equals(expected) or not (folder / 'grid.png').exists():
        raise ValueError(f'Incomplete run: {folder}')
    for i in range(len(pairs)):
        for suffix in ('pt','png'):
            if not (folder / f'image_{i:04d}.{suffix}').is_file():
                raise ValueError(f'Missing sample {i}: {folder}')
    return True


def run_pairing_experiments(split='validation'):
    if split not in {'validation','test'}:
        raise ValueError('Use validation or test')
    output, protocol, config, design = prepare_pairing_experiments()
    pairs = protocol['splits'][split]
    names = ['original']
    jobs = [('original', None)]
    for spec in design['specs']:
        for mode in (PAIRING_MODE,):
            name = spec['name'] + '_' + mode
            names.append(name)
            jobs.append((name, dict(removed=spec['removed'], looped=spec['looped'] if mode == 'damped' else [])))
    for name, settings in jobs:
        metadata = dict(experiment='pairings', design_sha256=design['contract']['source_sha256'], settings=settings)
        folder = output / split / name
        if pairing_complete(folder, config, pairs, metadata):
            print('Already complete:', name)
            continue
        transform = None if settings is None else lambda model, s=settings: build_pairing_model(model, s)
        run_variant(config, pairs, folder, model_transform=transform,
                    method_metadata=metadata, keep_checkpoint=KEEP_CHECKPOINTS)
    return output / split, names, design


def evaluate_pairing_experiments(split='validation'):
    if split not in {'validation', 'test'}:
        raise ValueError('Use validation or test')
    # All generation should finish before metric package installation, as in section 8.
    output = Path(RUN_ARGS.output) / 'pairing_experiments'
    design = json.loads((output / 'design.json').read_text())
    names = ['original'] + [s['name']+'_'+m for s in design['specs'] for m in (PAIRING_MODE,)]
    split_root = output / split
    compare_results(split_root, names)
    baseline = json.loads((split_root / 'original/manifest.json').read_text())
    comparison_grid(split_root, names, baseline['pairs'], count=1)
    reference, description = prepare_imagenette_reference(
        IMAGENETTE_SOURCE_ROOT, IMAGENETTE_PREPARED_ROOT, baseline['pairs'],
        images_per_class=FID_REAL_IMAGES_PER_CLASS, image_size=baseline['config']['image_size'], seed=0)
    if _extra_version('clean-fid') == 'not-installed':
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'clean-fid', 'scipy<1.18'], check=True)
    result = evaluate_fid_tpips(split_root, names, fid_reference_dir=reference,
        fid_reference_description=description, tpips_factors=(), fid_device=FID_DEVICE)
    display(result)
    print('Saved this mode only. Use section 9 after both teammates finish to compute recovery.')
    return result


def pairing_portable_config(config):
    # Local checkpoint location and device may differ; checkpoint SHA/protocol/code must match.
    return {k:v for k,v in config.items() if k not in {'device','checkpoint'}}


### 7B. RUN — this notebook's five variants and original baseline

Run validation first. Change `PAIRING_SPLIT` to `"test"` after freezing choices; selection remains fixed. The default test generates 5,000 images × six models. Finish generation and diagnostics before evaluation installs metric packages.


In [38]:
PAIRING_SPLIT = "validation"
pairing_root, pairing_names, pairing_design = run_pairing_experiments(PAIRING_SPLIT)
display(pd.DataFrame(pairing_design['specs']))
print('Saved new experiments:', pairing_root)


/Users/matthewlee/Desktop/CORNELL/CS5787/cs5787-deep-learning/venv/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Profiled label=0, seed=10000
Profiled label=217, seed=10001
Profiled label=482, seed=10002
Profiled label=491, seed=10003
Profiled label=497, seed=10004
Profiled label=566, seed=10005
Profiled label=569, seed=10006
Profiled label=571, seed=10007
Profiled label=574, seed=10008
Profiled label=701, seed=10009
Profiled label=0, seed=10010
Profiled label=217, seed=10011
Profiled label=482, seed=10012
Profiled label=491, seed=10013
Profiled label=497, seed=10014
Profiled label=566, seed=10015
Profiled label=569, seed=10016
Profiled label=571, seed=10017
Profiled label=574, seed=10018
Profiled label=701, seed=10019


/Users/matthewlee/Desktop/CORNELL/CS5787/cs5787-deep-learning/venv/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


original: 1/20, 7.30 s
original: 2/20, 7.29 s
original: 3/20, 7.41 s
original: 4/20, 7.41 s
original: 5/20, 7.36 s
original: 6/20, 7.29 s
original: 7/20, 7.28 s
original: 8/20, 7.30 s
original: 9/20, 7.37 s
original: 10/20, 7.31 s
original: 11/20, 7.30 s
original: 12/20, 7.29 s
original: 13/20, 7.41 s
original: 14/20, 7.50 s
original: 15/20, 7.30 s
original: 16/20, 7.51 s
original: 17/20, 7.32 s
original: 18/20, 7.33 s
original: 19/20, 7.37 s
original: 20/20, 7.42 s


/Users/matthewlee/Desktop/CORNELL/CS5787/cs5787-deep-learning/venv/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


original: 1/20, 7.04 s
original: 2/20, 7.24 s
original: 3/20, 7.04 s
original: 4/20, 7.12 s
original: 5/20, 7.25 s
original: 6/20, 7.23 s
original: 7/20, 7.04 s
original: 8/20, 7.29 s
original: 9/20, 7.04 s
original: 10/20, 7.04 s
original: 11/20, 7.04 s
original: 12/20, 7.29 s
original: 13/20, 7.06 s
original: 14/20, 7.03 s
original: 15/20, 7.03 s
original: 16/20, 7.26 s
original: 17/20, 7.13 s
original: 18/20, 7.22 s
original: 19/20, 7.05 s
original: 20/20, 7.05 s


... (6 outputs left out)

/Users/matthewlee/Desktop/CORNELL/CS5787/cs5787-deep-learning/venv/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


original: 1/20, 6.81 s
original: 2/20, 6.80 s
original: 3/20, 6.80 s
original: 4/20, 6.86 s
original: 5/20, 7.01 s
original: 6/20, 7.01 s
original: 7/20, 6.82 s
original: 8/20, 7.00 s
original: 9/20, 7.01 s
original: 10/20, 6.82 s
original: 11/20, 6.81 s
original: 12/20, 7.04 s
original: 13/20, 7.00 s
original: 14/20, 6.82 s
original: 15/20, 7.01 s
original: 16/20, 7.01 s
original: 17/20, 7.02 s
original: 18/20, 6.81 s
original: 19/20, 7.04 s
original: 20/20, 7.05 s


,name,group,removed,looped
0,feature_before,feature,[25],[26]
1,feature_after,feature,[27],[26]
2,position_late,position,"[22, 23]","[20, 21]"
3,position_middle,position,"[16, 17]","[14, 15]"
4,position_early,position,"[11, 12]","[9, 10]"


Saved new experiments: /Users/matthewlee/Desktop/CORNELL/CS5787/cs5787-deep-learning/team_runs/pairing_pruned_20261124T075035681906Z/pairing_experiments/validation


### 7C. DISPLAY — one sample's patch attention and block-ablation heatmaps

Diagnostics run outside measured inference. Both modes visualize the removed blocks using original-model attention and whole-trajectory single-block ablations. The damped notebook additionally captures actual second-execution attention for looped blocks; the pruned notebook never runs a damped variant. Attention is averaged over heads and queries on the conditional CFG branch at the middle sampling step. Ablation maps are final absolute RGB differences pooled to the 16×16 patch grid. They measure original-model sensitivity, not joint-removal recovery.


In [39]:
# PAIRING DIAGNOSTICS — one fixed experiment sample, outside measured inference.
from contextlib import contextmanager
import matplotlib.pyplot as plt
from torch.nn import functional as F


@contextmanager
def pairing_attention_capture(model, selected, sampling_step):
    """Read Q/K from the actual attention input; keep conditional CFG maps only."""
    records, handles, counts = {}, [], {}
    state = dict(step=-1, timestep=None)

    def before_timestep(module, args):
        state['step'] += 1
        state['timestep'] = int(args[0][0].item())
        counts.clear()

    def attention_hook(index):
        def hook(attn, args):
            counts[index] = counts.get(index, 0) + 1
            if state['step'] != sampling_step:
                return
            # Official timm Attention: [B,N,C] -> [3,B,heads,N,head_dim].
            x = args[0][:1]  # conditional branch of this single CFG sample
            b, n, c = x.shape
            qkv = attn.qkv(x).reshape(b, n, 3, attn.num_heads, c // attn.num_heads).permute(2,0,3,1,4)
            q, k = qkv[0], qkv[1]
            q = getattr(attn, 'q_norm', nn.Identity())(q)
            k = getattr(attn, 'k_norm', nn.Identity())(k)
            weights = ((q.float() * attn.scale) @ k.float().transpose(-2,-1)).softmax(dim=-1)
            side = math.isqrt(n)
            if side * side != n:
                raise ValueError('Expected square DiT patch grid without a CLS token')
            # Attention received by each key patch, averaged over heads and queries.
            records[(index, counts[index])] = dict(
                map=weights.mean(dim=(0,1,2)).reshape(side,side).cpu(),
                timestep=state['timestep'])
        return hook

    try:
        handles.append(model.t_embedder.register_forward_pre_hook(before_timestep))
        for index in sorted(set(selected)):
            if str(index) in model.blocks.retained:
                handles.append(model.blocks.retained[str(index)].attn.register_forward_pre_hook(attention_hook(index)))
        yield records
    finally:
        for handle in handles:
            handle.remove()


@torch.inference_mode()
def diagnose_pairing_experiments(split='validation', sample_index=0, sampling_step=None):
    if split not in {'validation', 'test'}:
        raise ValueError('Use validation or test')
    output = Path(RUN_ARGS.output) / 'pairing_experiments'
    design = json.loads((output / 'design.json').read_text())
    split_root = output / split
    baseline = json.loads((split_root / 'original/manifest.json').read_text())
    config, pairs = baseline['config'], baseline['pairs']
    if not 0 <= sample_index < len(pairs):
        raise ValueError('sample_index outside experiment samples')
    if sampling_step is None:
        sampling_step = int(config['sampling_steps']) // 2
    if not 0 <= sampling_step < int(config['sampling_steps']):
        raise ValueError('sampling_step outside diffusion trajectory')
    label, seed = pairs[sample_index]
    folder = split_root / 'diagnostics' / f'sample_{sample_index:04d}'
    folder.mkdir(parents=True, exist_ok=True)
    selected = sorted({i for s in design['specs'] for key in (('removed','looped') if PAIRING_MODE == 'damped' else ('removed',)) for i in s[key]})
    device = select_device(config['device'])
    model, vae, diffusion = _prepare(config, 'original', (), 1)
    try:
        model.to(device); vae.to(device)
        original_blocks = model.blocks
        with pairing_attention_capture(model, selected, sampling_step) as original_attention:
            original_image = generate(model, vae, diffusion, label, seed, config['cfg_scale'])
        # Ensure the diagnostic is actually the saved experiment's sample.
        saved = torch.load(split_root / 'original' / f'image_{sample_index:04d}.pt', weights_only=True)
        torch.testing.assert_close(original_image, saved, rtol=1e-4, atol=1e-5)
        maps, replay_attention = {}, {}
        for index in selected:
            # Counterfactual whole-trajectory ablation, one original block at a time.
            model.blocks = PairingBlocks(original_blocks.retained.values(), [index], [])
            ablated = generate(model, vae, diffusion, label, seed, config['cfg_scale'])
            error = (ablated-original_image).abs().mean(dim=1, keepdim=True)
            maps[index] = F.adaptive_avg_pool2d(error, (16,16))[0,0].cpu()
        if PAIRING_MODE == 'damped':
            for spec in design['specs']:
                model.blocks = PairingBlocks(original_blocks.retained.values(), spec['removed'], spec['looped'])
                with pairing_attention_capture(model, spec['looped'], sampling_step) as attention:
                    generated = generate(model, vae, diffusion, label, seed, config['cfg_scale'])
                saved = torch.load(split_root / (spec['name']+'_'+PAIRING_MODE) / f'image_{sample_index:04d}.pt', weights_only=True)
                torch.testing.assert_close(generated, saved, rtol=1e-4, atol=1e-5)
                replay_attention[spec['name']] = attention
        model.blocks = original_blocks
    finally:
        del model, vae, diffusion
        _cleanup()
    # Fixed color scales across all pairings; no timing measurements use these hooks.
    attention_values = [v['map'] for v in original_attention.values()]
    attention_values += [v['map'] for a in replay_attention.values() for v in a.values()]
    attention_max = max(float(v.max()) for v in attention_values)
    ablation_max = max(float(v.max()) for v in maps.values())
    torch.save(dict(original_attention=original_attention, replay_attention=replay_attention,
                    ablation=maps), folder / 'maps.pt')
    write_json(folder / 'diagnostics.json', dict(sample_index=sample_index, label=label, seed=seed,
        sampling_step=sampling_step, selected=selected,
        attention='conditional branch; mean over heads and query patches of softmax(QK^T/sqrt(d)); received attention',
        ablation='mean absolute RGB difference from original after skipping one block throughout the same seeded trajectory; pooled to 16x16',
        removed_attention='original model, first execution', looped_attention=('damped variant, second execution' if PAIRING_MODE == 'damped' else 'not applicable; no loops'),
        caveat='Single-block ablations measure original-model sensitivity, not joint pruning recovery.'))
    for spec in design['specs']:
        columns = [('removed', i) for i in spec['removed']] + ([('looped', i) for i in spec['looped']] if PAIRING_MODE == 'damped' else [])
        fig, axes = plt.subplots(2, len(columns)+1, figsize=(3*(len(columns)+1), 6), squeeze=False, constrained_layout=True)
        axes[0,0].imshow((original_image[0].permute(1,2,0).numpy()+1)/2)
        axes[0,0].set_title(f'Original: label {label}, seed {seed}')
        saved = torch.load(split_root / (spec['name']+'_'+PAIRING_MODE) / f'image_{sample_index:04d}.pt', weights_only=True)
        axes[1,0].imshow((saved[0].permute(1,2,0).numpy()+1)/2)
        axes[1,0].set_title(PAIRING_MODE.title() + ' output')
        for column, (role, index) in enumerate(columns, start=1):
            record = (original_attention[(index,1)] if role == 'removed'
                      else replay_attention[spec['name']][(index,2)])
            a = axes[0,column].imshow(record['map'], vmin=0, vmax=attention_max, cmap='viridis')
            axes[0,column].set_title(f'{role} block {index}\nattention, t={record["timestep"]}')
            b = axes[1,column].imshow(maps[index], vmin=0, vmax=ablation_max, cmap='magma')
            axes[1,column].set_title(f'Original minus ablated {index}\nabsolute RGB difference')
        for ax in axes.flat:
            ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(a, ax=list(axes[0,1:]), label='Mean attention received')
        fig.colorbar(b, ax=list(axes[1,1:]), label='Mean absolute error in [-1,1] RGB')
        fig.suptitle(spec['name'])
        path = folder / (spec['name']+'.png')
        fig.savefig(path, dpi=160)
        plt.close(fig)
        display(Image(filename=str(path)))
    return folder


In [40]:
#7B2

PAIRING_SPLIT = "test"

pairing_root, pairing_names, pairing_design = run_pairing_experiments(
    PAIRING_SPLIT
)

print("Test generation complete. Images saved to:", pairing_root)
display(pd.DataFrame(pairing_design["specs"]))

pairing_diagnostic_dir = diagnose_pairing_experiments(
    PAIRING_SPLIT,
    sample_index=0,
)

print("Test diagnostics saved to:", pairing_diagnostic_dir)

/Users/matthewlee/Desktop/CORNELL/CS5787/cs5787-deep-learning/venv/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


original: 1/5000, 7.51 s
original: 2/5000, 7.55 s
original: 3/5000, 7.47 s
original: 4/5000, 7.52 s
original: 5/5000, 7.51 s
original: 6/5000, 7.50 s
original: 7/5000, 7.44 s
original: 8/5000, 7.51 s
original: 9/5000, 7.56 s
original: 10/5000, 7.53 s
original: 11/5000, 7.50 s
original: 12/5000, 7.54 s
original: 13/5000, 7.57 s
original: 14/5000, 7.51 s
original: 15/5000, 7.51 s
original: 16/5000, 7.55 s
original: 17/5000, 7.53 s
original: 18/5000, 7.52 s
original: 19/5000, 7.51 s
original: 20/5000, 7.51 s
original: 21/5000, 7.42 s
original: 22/5000, 7.49 s
original: 23/5000, 7.50 s
original: 24/5000, 7.49 s
original: 25/5000, 7.48 s
original: 26/5000, 7.50 s
original: 27/5000, 7.48 s
original: 28/5000, 7.48 s
original: 29/5000, 7.50 s
original: 30/5000, 7.29 s
original: 31/5000, 7.51 s
original: 32/5000, 7.51 s
original: 33/5000, 7.50 s
original: 34/5000, 7.43 s
original: 35/5000, 7.53 s
original: 36/5000, 7.50 s
original: 37/5000, 7.52 s
original: 38/5000, 7.50 s
original: 39/5000, 7.

/Users/matthewlee/Desktop/CORNELL/CS5787/cs5787-deep-learning/venv/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


original: 1/5000, 7.04 s
original: 2/5000, 7.10 s
original: 3/5000, 7.02 s
original: 4/5000, 7.02 s
original: 5/5000, 7.08 s
original: 6/5000, 7.01 s
original: 7/5000, 7.02 s
original: 8/5000, 7.02 s
original: 9/5000, 7.10 s
original: 10/5000, 7.04 s
original: 11/5000, 7.02 s
original: 12/5000, 7.03 s
original: 13/5000, 7.03 s
original: 14/5000, 7.02 s
original: 15/5000, 7.04 s
original: 16/5000, 7.01 s
original: 17/5000, 7.03 s
original: 18/5000, 7.02 s
original: 19/5000, 7.15 s
original: 20/5000, 7.16 s
original: 21/5000, 7.02 s
original: 22/5000, 7.03 s
original: 23/5000, 7.02 s
original: 24/5000, 7.02 s
original: 25/5000, 7.06 s
original: 26/5000, 7.02 s
original: 27/5000, 7.02 s
original: 28/5000, 7.03 s
original: 29/5000, 7.23 s
original: 30/5000, 7.24 s
original: 31/5000, 7.22 s
original: 32/5000, 7.03 s
original: 33/5000, 7.02 s
original: 34/5000, 7.13 s
original: 35/5000, 7.03 s
original: 36/5000, 7.19 s
original: 37/5000, 7.03 s
original: 38/5000, 7.03 s
original: 39/5000, 7.

/Users/matthewlee/Desktop/CORNELL/CS5787/cs5787-deep-learning/venv/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


original: 1/5000, 7.16 s
original: 2/5000, 7.09 s
original: 3/5000, 7.05 s
original: 4/5000, 7.20 s
original: 5/5000, 7.23 s
original: 6/5000, 7.22 s
original: 7/5000, 7.16 s
original: 8/5000, 7.20 s
original: 9/5000, 7.12 s
original: 10/5000, 7.04 s
original: 11/5000, 7.22 s
original: 12/5000, 7.20 s
original: 13/5000, 7.20 s
original: 14/5000, 7.20 s
original: 15/5000, 7.19 s
original: 16/5000, 7.19 s
original: 17/5000, 7.18 s
original: 18/5000, 7.23 s
original: 19/5000, 7.23 s
original: 20/5000, 7.21 s
original: 21/5000, 7.03 s
original: 22/5000, 7.23 s
original: 23/5000, 7.03 s
original: 24/5000, 7.03 s
original: 25/5000, 7.05 s
original: 26/5000, 7.22 s
original: 27/5000, 7.24 s
original: 28/5000, 7.22 s
original: 29/5000, 7.21 s
original: 30/5000, 7.25 s
original: 31/5000, 7.12 s
original: 32/5000, 7.22 s
original: 33/5000, 7.20 s
original: 34/5000, 7.20 s
original: 35/5000, 7.16 s
original: 36/5000, 7.27 s
original: 37/5000, 7.21 s
original: 38/5000, 7.17 s
original: 39/5000, 7.

/Users/matthewlee/Desktop/CORNELL/CS5787/cs5787-deep-learning/venv/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


original: 1/5000, 6.84 s
original: 2/5000, 6.96 s
original: 3/5000, 6.79 s
original: 4/5000, 6.94 s
original: 5/5000, 6.78 s
original: 6/5000, 6.77 s
original: 7/5000, 6.93 s
original: 8/5000, 6.78 s
original: 9/5000, 6.77 s
original: 10/5000, 6.84 s
original: 11/5000, 6.77 s
original: 12/5000, 6.78 s
original: 13/5000, 6.77 s
original: 14/5000, 6.78 s
original: 15/5000, 6.78 s
original: 16/5000, 6.76 s
original: 17/5000, 6.92 s
original: 18/5000, 6.77 s
original: 19/5000, 6.94 s
original: 20/5000, 6.77 s
original: 21/5000, 6.95 s
original: 22/5000, 6.77 s
original: 23/5000, 6.77 s
original: 24/5000, 6.77 s
original: 25/5000, 6.93 s
original: 26/5000, 6.81 s
original: 27/5000, 6.78 s
original: 28/5000, 6.95 s
original: 29/5000, 6.77 s
original: 30/5000, 6.78 s
original: 31/5000, 6.77 s
original: 32/5000, 6.95 s
original: 33/5000, 6.95 s
original: 34/5000, 6.93 s
original: 35/5000, 6.77 s
original: 36/5000, 6.86 s
original: 37/5000, 6.77 s
original: 38/5000, 6.94 s
original: 39/5000, 6.

KeyboardInterrupt: 

In [ ]:
pairing_diagnostic_dir = diagnose_pairing_experiments(PAIRING_SPLIT, sample_index=0)


### 7D. EVALUATE — this notebook's LPIPS and FID

Evaluates only the five assigned variants plus the local original baseline. LPIPS uses matched original images; FID uses the same class-balanced Imagenette reference protocol in both notebooks. The 20-image validation FID is a noisy pilot. Aggregate and per-image metrics are saved in this run's split directory. Recovery relative to pruning is calculated in section 9 after both run directories are available.


In [ ]:
pairing_results = evaluate_pairing_experiments(PAIRING_SPLIT)


## 8. EXPORT — optional run archive

Includes frozen inputs, this notebook’s generation outputs, metric CSVs, diagnostics and manifests.


In [ ]:
# EXPORT — optional; no inference.
EXPORT_RUN = False
if EXPORT_RUN:
    import shutil
    archive = shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT.parent,
                                  base_dir=RUN_ROOT.name)
    print("Archive:", archive)
    if "google.colab" in sys.modules:
        from google.colab import files
        files.download(archive)
else:
    print("Set EXPORT_RUN=True to archive:", RUN_ROOT)


## 9. COMPARE — combine the two completed team runs (optional)

Set both paths to the top-level `RUN_ROOT` folders (not their `pairing_experiments` subfolders). This needs both teammates' saved images and metric files, but does not generate images. It verifies the five block pairings, protocol, labels/seeds and FID reference/settings. It recomputes LPIPS for both modes against the damped run's original images, then exports `pruned − damped` recovery (positive means improvement) for LPIPS and FID. Hardware timings are not pooled. Output: `DAMPED_RUN_ROOT/pairing_experiments/<split>/team_comparison/`.


In [ ]:
def compare_team_pairings(damped_run, pruned_run, split='validation', metric=None):
    """Read both run roots; use ONE original reference for all paired LPIPS."""
    if split not in {'validation', 'test'}:
        raise ValueError('Use validation or test')
    roots = {m: Path(p).expanduser().resolve() / 'pairing_experiments'
             for m,p in [('damped',damped_run),('pruned',pruned_run)]}
    designs = {m: json.loads((p/'design.json').read_text()) for m,p in roots.items()}
    for mode, design in designs.items():
        if design['contract']['mode'] != mode:
            raise ValueError(f'Wrong notebook run supplied for {mode}')
    left, right = designs['damped'], designs['pruned']
    if (left['specs'] != right['specs'] or left['anchor'] != right['anchor']
            or left['contract']['source_sha256'] != right['contract']['source_sha256']):
        raise ValueError('Different block selections or experiment code; share design.json and rerun')
    if pairing_portable_config(left['contract']['config']) != pairing_portable_config(right['contract']['config']):
        raise ValueError('Different generation protocols/settings')
    summaries, manifests, evaluations = {}, {}, {}
    for mode, root in roots.items():
        folder = root/split
        manifests[mode] = json.loads((folder/'original/manifest.json').read_text())
        evaluations[mode] = json.loads((folder/'evaluation_extended.json').read_text())
        summaries[mode] = pd.read_csv(folder/'results_extended.csv').set_index('variant', verify_integrity=True)
    baseline = manifests['damped']
    if baseline['pairs'] != manifests['pruned']['pairs']:
        raise ValueError('Different evaluation labels/seeds')
    for mode in roots:
        if pairing_portable_config(manifests[mode]['config']) != pairing_portable_config(left['contract']['config']):
            raise ValueError('Baseline does not match frozen design')
    # Require equivalent real-image FID reference and feature extraction settings.
    fids = [evaluations[m]['fid'] for m in ('damped','pruned')]
    keys = ('implementation','mode','features','reference_images','generated_images','reference_inventory_sha256')
    if any(f is None for f in fids) or any(fids[0][k] != fids[1][k] for k in keys):
        raise ValueError('FID reference or feature settings differ')
    for package in ('clean-fid','torchvision','pillow','numpy','scipy'):
        if evaluations['damped']['packages'].get(package) != evaluations['pruned']['packages'].get(package):
            raise ValueError(f'FID package version differs: {package}')
    if metric is None:
        import lpips
        metric = lpips.LPIPS(net='alex', version='0.1').cpu().eval()
    rows, paired_rows = [], []
    with torch.inference_mode():
        for spec in left['specs']:
            folders = {m: roots[m]/split/(spec['name']+'_'+m) for m in roots}
            for mode, folder in folders.items():
                manifest = json.loads((folder/'manifest.json').read_text())
                settings = dict(removed=spec['removed'], looped=spec['looped'] if mode=='damped' else [])
                if (manifest['pairs'] != baseline['pairs']
                        or pairing_portable_config(manifest['config']) != pairing_portable_config(baseline['config'])
                        or manifest['method_metadata']['settings'] != settings):
                    raise ValueError(f'Variant does not match pairing: {folder}')
                expected = pd.DataFrame([(i,*p) for i,p in enumerate(baseline['pairs'])], columns=['index','label','seed'])
                if not pd.read_csv(folder/'samples.csv')[['index','label','seed']].equals(expected):
                    raise ValueError(f'Incomplete sample set: {folder}')
            values = {m: [] for m in roots}
            for index, (label,seed) in enumerate(baseline['pairs']):
                reference = torch.load(roots['damped']/split/'original'/f'image_{index:04d}.pt', weights_only=True, map_location='cpu')
                reference = preprocess_image(reference, baseline['config']['image_size'])
                pair = dict(pairing=spec['name'], index=index, label=label, seed=seed)
                for mode, folder in folders.items():
                    image = torch.load(folder/f'image_{index:04d}.pt', weights_only=True, map_location='cpu')
                    image = preprocess_image(image, baseline['config']['image_size'])
                    score = float(metric(reference,image).item())
                    if not np.isfinite(score):
                        raise ValueError('Nonfinite LPIPS')
                    values[mode].append(score)
                    pair['lpips_'+mode] = score
                pair['lpips_recovered'] = pair['lpips_pruned'] - pair['lpips_damped']
                paired_rows.append(pair)
            row = dict(pairing=spec['name'], group=spec['group'], removed=spec['removed'], looped=spec['looped'])
            for mode in roots:
                saved = summaries[mode].loc[spec['name']+'_'+mode]
                if int(saved['images']) != len(baseline['pairs']) or not np.isfinite(saved['fid']):
                    raise ValueError('Invalid saved FID results')
                row['lpips_mean_'+mode] = float(np.mean(values[mode]))
                row['fid_'+mode] = float(saved['fid'])
            for key in ('lpips_mean','fid'):
                row[key+'_recovered'] = row[key+'_pruned'] - row[key+'_damped']
            rows.append(row)
    destination = roots['damped']/split/'team_comparison'
    destination.mkdir(exist_ok=True)
    result = pd.DataFrame(rows)
    result.to_csv(destination/'looping_recovery.csv', index=False)
    pd.DataFrame(paired_rows).to_csv(destination/'paired_recovery.csv', index=False)
    write_json(destination/'comparison.json', dict(
        damped_run=str(roots['damped']), pruned_run=str(roots['pruned']), split=split,
        lpips_reference='damped run original images for BOTH methods; recomputed on CPU',
        metric='lpips_alex_v0.1' if metric.__class__.__module__.startswith('lpips') else 'injected_metric',
        fid='saved scores, verified reference/settings/package versions',
        runtime_damped=manifests['damped']['runtime'], runtime_pruned=manifests['pruned']['runtime'],
        interpretation='positive recovered means improvement; hardware timings are not pooled'))
    display(result)
    print('Saved:', destination)
    return result


In [ ]:
DAMPED_RUN_ROOT = None  # e.g. "/content/.../pairing_damped_<timestamp>"
PRUNED_RUN_ROOT = None  # e.g. "/content/.../pairing_pruned_<timestamp>"
if DAMPED_RUN_ROOT is not None and PRUNED_RUN_ROOT is not None:
    team_recovery = compare_team_pairings(DAMPED_RUN_ROOT, PRUNED_RUN_ROOT, PAIRING_SPLIT)
else:
    print("Optional: set both run roots after the two teammates finish.")
